In [ ]:
from pathlib import Path
import os
import sys
_release_root = Path(os.environ.get('AVATAR_RELEASE_ROOT', Path.cwd())).resolve()
while not (_release_root / 'avatar_paths.py').is_file():
    if _release_root == _release_root.parent:
        raise RuntimeError('Open this notebook from the code package directory.')
    _release_root = _release_root.parent
if str(_release_root) not in sys.path:
    sys.path.insert(0, str(_release_root))
from avatar_paths import project_path, get_value, get_column, get_path, get_label


In [ ]:
import os
import pandas as pd
folder_path = str(project_path(get_value('science.paths.directory_094', 'directory_094')))
height_file = [f for f in os.listdir(folder_path) if f.startswith('Height') and f.endswith('.txt')][0]
input_path = os.path.join(folder_path, height_file)
df = pd.read_csv(input_path, sep='\t', header=4, low_memory=False)
name_col = 'Metabolite name'
named = df[df[name_col].notna() & ~df[name_col].astype(str).str.contains('Unknown|w/o MS2', case=False, na=False)]
without_ms2 = df[df[name_col].astype(str).str.contains('w/o MS2', case=False, na=False)]
unknown = df[df[name_col].astype(str).str.contains('Unknown', case=False, na=False)]
random_rows = df.sample(n=min(150, len(df)), random_state=20260728)
parts = []
if len(named) > 0:
    parts.append(named.sample(n=min(100, len(named)), random_state=1))
if len(without_ms2) > 0:
    parts.append(without_ms2.sample(n=min(100, len(without_ms2)), random_state=2))
if len(unknown) > 0:
    parts.append(unknown.sample(n=min(100, len(unknown)), random_state=3))
parts.append(random_rows)
subset = pd.concat(parts, axis=0).drop_duplicates().reset_index(drop=True)
output_path = os.path.join(folder_path, get_value('science.tables.features_027', 'features_027.csv'))
subset.to_csv(output_path, index=False, encoding='utf-8-sig')
print('Original dimensions:', df.shape)
print('Subset dimensions:', subset.shape)



In [ ]:
import os
import re
import numpy as np
import pandas as pd
input_path = str(project_path(get_value('science.tables.features_028', 'features_028.txt')))
output_dir = str(project_path(get_value('science.paths.directory_095', 'directory_095')))
os.makedirs(output_dir, exist_ok=True)
TOP_N = 300
meta4 = pd.read_csv(input_path, sep='\t', header=None, nrows=4, dtype=str, keep_default_na=False, low_memory=False)
df = pd.read_csv(input_path, sep='\t', header=4, low_memory=False)
print('Data dimensions:', df.shape)
print('Dimensions of the first four metadata rows:', meta4.shape)
if meta4.shape[1] != df.shape[1]:
    raise ValueError(f'The first four rows have {meta4.shape[1]} columns, whereas the data have {df.shape[1]} columns. The column counts do not match; check whether Excel modified the file.')
column_metadata = pd.DataFrame({'column_name': df.columns.astype(str), 'class': meta4.iloc[0].astype(str).str.strip().values, 'file_type': meta4.iloc[1].astype(str).str.strip().values, 'injection_order': meta4.iloc[2].astype(str).str.strip().values, 'batch_id': meta4.iloc[3].astype(str).str.strip().values})
column_metadata['file_type_lower'] = column_metadata['file_type'].astype(str).str.lower().str.strip()
sample_cols = column_metadata.loc[column_metadata['file_type_lower'].eq('sample'), 'column_name'].tolist()
qc_cols_all = column_metadata.loc[column_metadata['file_type_lower'].eq('qc'), 'column_name'].tolist()
qc_ddms2_cols = [c for c in qc_cols_all if 'ddms2' in str(c).lower()]
qc_quant_cols = [c for c in qc_cols_all if 'ddms2' not in str(c).lower()]
blank_cols = column_metadata.loc[column_metadata['file_type_lower'].eq('blank'), 'column_name'].tolist()

print('Clinical samples:', len(sample_cols))
print('All QC samples:', len(qc_cols_all))
print('Routine quantitative QC samples:', len(qc_quant_cols))
print('ddMS2 QC:', len(qc_ddms2_cols))
print('Blank:', len(blank_cols))
column_metadata.drop(columns=['file_type_lower']).to_csv(os.path.join(output_dir, get_value('science.tables.result_119', 'result_119.csv')), index=False, encoding='utf-8-sig')
annotation_cols_wanted = ['Alignment ID', 'Average Rt(min)', 'Average Mz', 'Metabolite name', 'Adduct type', 'Post curation result', 'Fill %', 'MS/MS assigned', 'Reference RT', 'Reference m/z', 'Formula', 'Ontology', 'INCHIKEY', 'SMILES', 'Annotation tag (VS1.0)', 'RT matched', 'm/z matched', 'MS/MS matched', 'Comment', 'Manually modified for quantification', 'Manually modified for annotation', 'Isotope tracking parent ID', 'Isotope tracking weight number', 'RT similarity', 'm/z similarity', 'Simple dot product', 'Weighted dot product', 'Reverse dot product', 'Matched peaks count', 'Matched peaks percentage', 'Total score', 'S/N average', 'Spectrum reference file name', 'MS1 isotopic spectrum', 'MS/MS spectrum']
annotation_cols = [c for c in annotation_cols_wanted if c in df.columns]
missing_annotation_cols = [c for c in annotation_cols_wanted if c not in df.columns]
print('\nAnnotation columns found:', len(annotation_cols))
if missing_annotation_cols:
    print('Annotation columns not found:')
    print(missing_annotation_cols)
intensity_cols = list(dict.fromkeys(sample_cols + qc_quant_cols + blank_cols))
X_intensity = df[intensity_cols].apply(pd.to_numeric, errors='coerce')
X_detected = X_intensity.mask(X_intensity <= 0)
quality = pd.DataFrame(index=df.index)
sample_matrix = X_detected[sample_cols]
quality['Sample_detection_rate'] = sample_matrix.notna().mean(axis=1)
quality['Sample_median_detected'] = sample_matrix.median(axis=1, skipna=True)
quality['Sample_mean_detected'] = sample_matrix.mean(axis=1, skipna=True)
quality['Sample_max'] = sample_matrix.max(axis=1, skipna=True)
valid_batches = sorted([x for x in column_metadata['batch_id'].astype(str).unique() if x not in {'', 'NA', 'nan', 'Average', 'Stdev'}])
qc_pass_30_list = []
qc_pass_40_list = []
blank_pass_3_list = []
blank_pass_5_list = []
batch_qc_cv_cols = []
batch_blank_ratio_cols = []
for batch in valid_batches:
    batch_mask = column_metadata['batch_id'].astype(str).eq(str(batch))
    batch_sample_cols = column_metadata.loc[batch_mask & column_metadata['file_type_lower'].eq('sample'), 'column_name'].tolist()
    batch_qc_cols = column_metadata.loc[batch_mask & column_metadata['file_type_lower'].eq('qc'), 'column_name'].tolist()
    batch_qc_cols = [c for c in batch_qc_cols if 'ddms2' not in str(c).lower()]
    batch_blank_cols = column_metadata.loc[batch_mask & column_metadata['file_type_lower'].eq('blank'), 'column_name'].tolist()
    print(f'\nBatch {batch}: Sample={len(batch_sample_cols)}, quantitative QC={len(batch_qc_cols)}, Blank={len(batch_blank_cols)}')
    if len(batch_qc_cols) >= 3:
        qc_mat = X_detected[batch_qc_cols]
        qc_detection = qc_mat.notna().mean(axis=1)
        qc_mean = qc_mat.mean(axis=1, skipna=True)
        qc_sd = qc_mat.std(axis=1, skipna=True, ddof=1)
        qc_cv = qc_sd / qc_mean
        quality[f'QC_detection_B{batch}'] = qc_detection
        quality[f'QC_mean_B{batch}'] = qc_mean
        quality[f'QC_CV_B{batch}'] = qc_cv
        qc_pass_30 = (qc_detection >= 0.7) & (qc_cv <= 0.3)
        qc_pass_40 = (qc_detection >= 0.7) & (qc_cv <= 0.4)
        qc_pass_30_list.append(qc_pass_30)
        qc_pass_40_list.append(qc_pass_40)
        batch_qc_cv_cols.append(f'QC_CV_B{batch}')
    if len(batch_sample_cols) > 0 and len(batch_blank_cols) > 0:
        sample_batch_median = X_detected[batch_sample_cols].median(axis=1, skipna=True)
        blank_raw = X_intensity[batch_blank_cols].mean(axis=1, skipna=True)
        ratio = pd.Series(np.where(blank_raw > 0, sample_batch_median / blank_raw, np.inf), index=df.index)
        quality[f'Sample_median_B{batch}'] = sample_batch_median
        quality[f'Blank_mean_B{batch}'] = blank_raw
        quality[f'Sample_blank_ratio_B{batch}'] = ratio
        blank_pass_3_list.append(ratio >= 3)
        blank_pass_5_list.append(ratio >= 5)
        batch_blank_ratio_cols.append(f'Sample_blank_ratio_B{batch}')
if qc_pass_30_list:
    quality['QC_pass_all_batches_30pct'] = np.logical_and.reduce([x.fillna(False).values for x in qc_pass_30_list])
else:
    quality['QC_pass_all_batches_30pct'] = False
if qc_pass_40_list:
    quality['QC_pass_all_batches_40pct'] = np.logical_and.reduce([x.fillna(False).values for x in qc_pass_40_list])
else:
    quality['QC_pass_all_batches_40pct'] = False
if blank_pass_3_list:
    quality['Blank_pass_all_batches_ratio3'] = np.logical_and.reduce([x.fillna(False).values for x in blank_pass_3_list])
else:
    quality['Blank_pass_all_batches_ratio3'] = False
if blank_pass_5_list:
    quality['Blank_pass_all_batches_ratio5'] = np.logical_and.reduce([x.fillna(False).values for x in blank_pass_5_list])
else:
    quality['Blank_pass_all_batches_ratio5'] = False
if batch_qc_cv_cols:
    quality['QC_CV_max'] = quality[batch_qc_cv_cols].max(axis=1, skipna=True)
if batch_blank_ratio_cols:
    quality['Sample_blank_ratio_min'] = quality[batch_blank_ratio_cols].min(axis=1, skipna=True)
PROTON = 1.007276466621
SODIUM = 22.989218
POTASSIUM = 38.963158
AMMONIUM = 18.033823
WATER = 18.010564684

def estimate_neutral_mass(mz, adduct):
    try:
        mz = float(mz)
    except Exception:
        return np.nan
    adduct = str(adduct).replace(' ', '')
    rules = {'[M+H]+': mz - PROTON, '[M+Na]+': mz - SODIUM, '[M+K]+': mz - POTASSIUM, '[M+NH4]+': mz - AMMONIUM, '[M-H2O+H]+': mz + WATER - PROTON, '[M-2H2O+H]+': mz + 2 * WATER - PROTON, '[M]+': mz, '[M+2H]2+': 2 * mz - 2 * PROTON, '[M+H+Na]2+': 2 * mz - PROTON - SODIUM}
    return rules.get(adduct, np.nan)
average_mz = pd.to_numeric(df.get('Average Mz'), errors='coerce')
adduct_series = df.get('Adduct type', pd.Series('', index=df.index))
quality['Estimated_neutral_mass'] = [estimate_neutral_mass(mz, adduct) for mz, adduct in zip(average_mz, adduct_series)]
reference_mz = pd.to_numeric(df.get('Reference m/z'), errors='coerce')
quality['Reference_mz_error_ppm'] = np.where(reference_mz > 0, (average_mz - reference_mz) / reference_mz * 1000000.0, np.nan)
name = df.get('Metabolite name', pd.Series('', index=df.index)).fillna('').astype(str).str.strip()
name_lower = name.str.lower()
annotation_tag = pd.to_numeric(df.get('Annotation tag (VS1.0)'), errors='coerce')
msms_spectrum = df.get('MS/MS spectrum', pd.Series('', index=df.index)).fillna('').astype(str).str.strip()
ms1_spectrum = df.get('MS1 isotopic spectrum', pd.Series('', index=df.index)).fillna('').astype(str).str.strip()
formula = df.get('Formula', pd.Series('', index=df.index)).fillna('').astype(str).str.strip()
ontology = df.get('Ontology', pd.Series('', index=df.index)).fillna('').astype(str).str.strip()
adduct = df.get('Adduct type', pd.Series('', index=df.index)).fillna('').astype(str).str.strip()
is_without_ms2 = name_lower.str.contains('w/o\\s*ms2|w/o\\s*ms/ms|without\\s*ms', regex=True, na=False)
is_unknown = name.eq('') | name_lower.str.contains('unknown|unkown|unidentified|no annotation', regex=True, na=False)
is_low_annotation_tag = annotation_tag.isin([530, 999])
has_msms_spectrum = msms_spectrum.str.len() >= 10 & msms_spectrum.str.contains(':', regex=False)
has_ms1_spectrum = ms1_spectrum.str.len() >= 10 & ms1_spectrum.str.contains(':', regex=False)
has_formula = formula.ne('') & ~formula.str.lower().isin(['na', 'nan', 'none'])
has_ontology = ontology.ne('') & ~ontology.str.lower().isin(['na', 'nan', 'none'])
has_adduct = adduct.ne('') & ~adduct.str.lower().isin(['na', 'nan', 'none'])
candidate_mask = is_unknown | is_without_ms2 | is_low_annotation_tag
candidate_group = np.select([is_without_ms2, is_unknown & has_msms_spectrum, is_unknown & ~has_msms_spectrum, is_low_annotation_tag], ['w/o MS2 / weak annotation', 'Unknown with MS/MS', 'Unknown without MS/MS', 'Low-confidence annotation tag'], default='Other')
quality['Original_annotation_group'] = candidate_group
quality['Has_MS1_spectrum'] = has_ms1_spectrum
quality['Has_MSMS_spectrum'] = has_msms_spectrum
quality['Has_formula'] = has_formula
quality['Has_ontology'] = has_ontology
quality['Has_adduct'] = has_adduct
priority_score = pd.Series(0, index=df.index, dtype=float)
priority_score += has_msms_spectrum.astype(int) * 5
priority_score += has_ms1_spectrum.astype(int) * 1
priority_score += has_formula.astype(int) * 2
priority_score += has_ontology.astype(int) * 1
priority_score += has_adduct.astype(int) * 1
priority_score += quality['QC_pass_all_batches_30pct'].fillna(False).astype(int) * 3
priority_score += quality['QC_pass_all_batches_40pct'].fillna(False).astype(int) * 1
priority_score += (quality['Sample_detection_rate'] >= 0.5).fillna(False).astype(int) * 2
priority_score += quality['Blank_pass_all_batches_ratio5'].fillna(False).astype(int) * 2
quality['Reannotation_priority_score'] = priority_score
quality['Reannotation_priority_tier'] = np.select([candidate_mask & has_msms_spectrum & quality['QC_pass_all_batches_30pct'] & quality['Blank_pass_all_batches_ratio5'], candidate_mask & has_msms_spectrum & quality['QC_pass_all_batches_40pct'], candidate_mask & has_msms_spectrum, candidate_mask & ~has_msms_spectrum & (has_formula | has_ontology), candidate_mask], ['Tier A: MS/MS + strict QC', 'Tier B: MS/MS + rescue QC', 'Tier C: MS/MS available', 'Tier D: MS1/formula/class only', 'Tier E: limited evidence'], default='Not a reannotation candidate')
annotation_df = df[annotation_cols].copy()
result = pd.concat([annotation_df, quality], axis=1)
candidates = result.loc[candidate_mask].copy()
candidates = candidates.sort_values(by=['Reannotation_priority_score', 'Has_MSMS_spectrum', 'QC_pass_all_batches_30pct', 'Sample_detection_rate'], ascending=[False, False, False, False]).reset_index(drop=True)
candidates.insert(0, 'Reannotation_rank', np.arange(1, len(candidates) + 1))
spectrum_cols = ['MS1 isotopic spectrum', 'MS/MS spectrum']
compact_cols = [c for c in candidates.columns if c not in spectrum_cols]
compact_path = os.path.join(output_dir, get_value('science.tables.result_120', 'result_120.csv'))
candidates[compact_cols].to_csv(compact_path, index=False, encoding='utf-8-sig')
top_candidates = candidates.head(min(TOP_N, len(candidates))).copy()
top_path = os.path.join(output_dir, f"BloodPos_reannotation_priority_top{TOP_N}{get_value('science.tables.result_121', 'result_121.csv')}")
top_candidates.to_csv(top_path, index=False, encoding='utf-8-sig')
full_gzip_path = os.path.join(output_dir, get_value('science.tables.result_122', 'result_122.csv.gz'))
candidates.to_csv(full_gzip_path, index=False, encoding='utf-8-sig', compression='gzip')
summary = pd.DataFrame({get_column('science.translation.preprocess_003', 'Item'): [get_value('science.translation.preprocess_005', 'Original features'), get_value('science.translation.preprocess_006', 'Reannotation candidates'), get_value('science.translation.preprocess_007', 'Candidates with MS/MS spectra'), get_value('science.translation.preprocess_008', 'Candidates with molecular formulas'), get_value('science.translation.preprocess_009', 'Tier A candidates'), get_value('science.translation.preprocess_010', 'Tier B candidates'), get_value('science.translation.preprocess_011', 'Tier C candidates'), get_value('science.translation.preprocess_012', 'Tier D candidates'), get_value('science.translation.preprocess_013', 'Tier E candidates')], get_column('science.translation.preprocess_004', 'Count'): [len(df), len(candidates), int(candidates['Has_MSMS_spectrum'].sum()), int(candidates['Has_formula'].sum()), int(candidates['Reannotation_priority_tier'].str.startswith('Tier A').sum()), int(candidates['Reannotation_priority_tier'].str.startswith('Tier B').sum()), int(candidates['Reannotation_priority_tier'].str.startswith('Tier C').sum()), int(candidates['Reannotation_priority_tier'].str.startswith('Tier D').sum()), int(candidates['Reannotation_priority_tier'].str.startswith('Tier E').sum())]})
summary_path = os.path.join(output_dir, get_value('science.tables.metrics_055', 'metrics_055.csv'))
summary.to_csv(summary_path, index=False, encoding='utf-8-sig')

print(summary.to_string(index=False))








In [ ]:
import os
import numpy as np
import pandas as pd
input_path = str(project_path(get_value('science.tables.metrics_056', 'metrics_056.csv.gz')))
output_dir = str(project_path(get_value('science.paths.directory_095', 'directory_095')))
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, get_value('science.tables.result_123', 'result_123.csv.gz'))
df = pd.read_csv(input_path, compression='gzip', low_memory=False)
print('All candidates:', df.shape)
tier = df['Reannotation_priority_tier'].fillna('').astype(str)
keep = tier.str.startswith(('Tier A', 'Tier B'))
tier_ab = df.loc[keep].copy()
print('Tier A + B:', tier_ab.shape)

def to_bool(series):
    return series.fillna('').astype(str).str.strip().str.lower().isin(['true', '1', 'yes'])
if 'MS/MS assigned' in tier_ab.columns:
    tier_ab['MSMS_assigned_bool'] = to_bool(tier_ab['MS/MS assigned'])
else:
    tier_ab['MSMS_assigned_bool'] = False
if 'MS/MS matched' in tier_ab.columns:
    tier_ab['MSMS_matched_bool'] = to_bool(tier_ab['MS/MS matched'])
else:
    tier_ab['MSMS_matched_bool'] = False
msms = tier_ab['MS/MS spectrum'].fillna('').astype(str).str.strip()
tier_ab['Has_valid_MSMS_string'] = msms.str.contains(':', regex=False) & (msms.str.len() >= 20)
sort_columns = ['MSMS_matched_bool', 'MSMS_assigned_bool', 'Has_valid_MSMS_string', 'Reannotation_priority_score', 'Sample_detection_rate']
sort_columns = [c for c in sort_columns if c in tier_ab.columns]
tier_ab = tier_ab.sort_values(sort_columns, ascending=[False] * len(sort_columns)).reset_index(drop=True)
tier_ab.insert(0, 'Export_rank', np.arange(1, len(tier_ab) + 1))
wanted_columns = ['Export_rank', 'Reannotation_rank', 'Alignment ID', 'Average Rt(min)', 'Average Mz', 'Metabolite name', 'Adduct type', 'Formula', 'Ontology', 'INCHIKEY', 'SMILES', 'Annotation tag (VS1.0)', 'MS/MS assigned', 'MS/MS matched', 'MSMS_assigned_bool', 'MSMS_matched_bool', 'Has_valid_MSMS_string', 'RT similarity', 'm/z similarity', 'Simple dot product', 'Weighted dot product', 'Reverse dot product', 'Matched peaks count', 'Matched peaks percentage', 'Total score', 'S/N average', 'MS1 isotopic spectrum', 'MS/MS spectrum', 'Estimated_neutral_mass', 'Reference_mz_error_ppm', 'QC_CV_max', 'Sample_detection_rate', 'Sample_blank_ratio_min', 'QC_pass_all_batches_30pct', 'QC_pass_all_batches_40pct', 'Blank_pass_all_batches_ratio5', 'Original_annotation_group', 'Reannotation_priority_score', 'Reannotation_priority_tier']
existing_columns = [c for c in wanted_columns if c in tier_ab.columns]
tier_ab = tier_ab[existing_columns]
tier_ab.to_csv(output_path, index=False, encoding='utf-8-sig', compression='gzip')


print('Final dimensions:', tier_ab.shape)



In [ ]:
import os
import math
import pandas as pd
input_path = str(project_path(get_value('science.tables.metrics_057', 'metrics_057.csv.gz')))
output_dir = os.path.dirname(input_path)
df = pd.read_csv(input_path, compression='gzip', low_memory=False)
n_parts = 4
chunk_size = math.ceil(len(df) / n_parts)
for i in range(n_parts):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, len(df))
    part = df.iloc[start:end].copy()
    if part.empty:
        continue
    output_path = os.path.join(output_dir, f'BloodPos_TierAB_with_spectra_part{i + 1}.csv.gz')
    part.to_csv(output_path, index=False, encoding='utf-8-sig', compression='gzip')
    


In [ ]:
from pathlib import Path
import math
import re
import numpy as np
import pandas as pd
bloodneg_dir = Path(str(project_path(get_value('science.paths.directory_096', 'directory_096'))))
manual_input_path = None
output_dir = bloodneg_dir / get_value('science.paths.directory_097', 'directory_097')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / get_value('science.tables.result_124', 'result_124.csv.gz')
summary_path = output_dir / get_value('science.tables.metrics_058', 'metrics_058.csv')
split_threshold_mb = 20
n_split_parts = 4
sample_detection_threshold = 0.5
qc_detection_threshold = 0.7
strict_qc_cv_threshold = 0.3
rescue_qc_cv_threshold = 0.4
sample_blank_ratio_threshold = 5.0
minimum_msms_peak_count = 3
if manual_input_path is not None:
    input_path = Path(manual_input_path)
else:
    height_candidates = list(bloodneg_dir.glob('Height_*.txt'))
    if not height_candidates:
        raise FileNotFoundError(f'No Height_*.txt file was found in this directory:\n{bloodneg_dir}')
    input_path = max(height_candidates, key=lambda path: path.stat().st_size)
if not input_path.exists():
    raise FileNotFoundError(input_path)
print('Blood NEG Height file used:')


metadata_raw = pd.read_csv(input_path, sep='\t', header=None, nrows=4, dtype=str, keep_default_na=False, engine='python')
data = pd.read_csv(input_path, sep='\t', header=4, low_memory=False)
metadata_raw = metadata_raw.reindex(columns=range(len(data.columns)), fill_value='')
metadata_raw.columns = data.columns
metadata_raw.index = ['Class', 'File type', 'Injection order', 'Batch ID']
print('\nOriginal data dimensions:', data.shape)

def normalize_column_name(value):
    return re.sub('[^a-z0-9]+', '', str(value).strip().lower())
normalized_column_map = {normalize_column_name(column): column for column in data.columns}

def find_column(candidates, required=True):
    for candidate in candidates:
        key = normalize_column_name(candidate)
        if key in normalized_column_map:
            return normalized_column_map[key]
    if required:
        raise KeyError('None of the following columns was found:\n' + '\n'.join(candidates))
    return None
alignment_id_col = find_column(['Alignment ID', 'AlignmentID'])
rt_col = find_column(['Average Rt(min)', 'Average RT(min)', 'Average Rt'])
mz_col = find_column(['Average Mz', 'Average m/z', 'AverageMz'])
name_col = find_column(['Metabolite name', 'Metabolite Name'])
adduct_col = find_column(['Adduct type', 'Adduct'])
formula_col = find_column(['Formula'], required=False)
msms_spectrum_col = find_column(['MS/MS spectrum', 'MSMS spectrum'], required=False)
ms1_spectrum_col = find_column(['MS1 isotopic spectrum', 'MS1 isotope spectrum'], required=False)
msms_assigned_col = find_column(['MS/MS assigned', 'MSMS assigned'], required=False)
msms_matched_col = find_column(['MS/MS matched', 'MSMS matched'], required=False)
file_type = metadata_raw.loc['File type'].fillna('').astype(str).str.strip().str.lower()
batch_id = metadata_raw.loc['Batch ID'].fillna('').astype(str).str.strip()

def normalize_batch(value):
    value = str(value).strip()
    if not value or value.lower() in {'nan', 'none', 'na'}:
        return 'ALL'
    try:
        numeric = float(value)
        if numeric.is_integer():
            return str(int(numeric))
    except ValueError:
        pass
    return value
batch_id = batch_id.map(normalize_batch)
sample_columns = [column for column in data.columns if file_type.get(column, '') in {'sample', 'biological sample', 'study sample'}]
qc_columns_all = [column for column in data.columns if 'qc' in file_type.get(column, '')]
qc_columns = [column for column in qc_columns_all if 'ddms2' not in str(column).lower()]
blank_columns = [column for column in data.columns if 'blank' in file_type.get(column, '')]

print('Clinical/biological samples:', len(sample_columns))
print('All QC samples:', len(qc_columns_all))
print('Quantitative QC samples:', len(qc_columns))
print('Blank:', len(blank_columns))
if not sample_columns:
    print('\nUnique values in the File type row:')
    print(sorted(file_type.unique()))
    raise ValueError('No Sample columns were identified; check File type in the first four rows.')
if len(qc_columns) < 2:
    raise ValueError('Fewer than two QC columns are available for quantitative QC CV calculation.')
intensity_columns = list(dict.fromkeys(sample_columns + qc_columns + blank_columns))
intensity = data[intensity_columns].apply(pd.to_numeric, errors='coerce')
intensity = intensity.mask(intensity <= 0, np.nan)
sample_detection_rate = intensity[sample_columns].notna().mean(axis=1)
sample_median_all = intensity[sample_columns].median(axis=1, skipna=True)
sample_batches = sorted({batch_id[column] for column in sample_columns})
qc_cv_by_batch = {}
qc_detection_by_batch = {}
qc_count_by_batch = {}
for batch in sample_batches:
    batch_qc_columns = [column for column in qc_columns if batch_id[column] == batch]
    if not batch_qc_columns and len(sample_batches) == 1:
        batch_qc_columns = qc_columns.copy()
    qc_count_by_batch[batch] = len(batch_qc_columns)
    if len(batch_qc_columns) >= 2:
        batch_qc = intensity[batch_qc_columns]
        qc_mean = batch_qc.mean(axis=1, skipna=True)
        qc_sd = batch_qc.std(axis=1, skipna=True, ddof=1)
        qc_cv = qc_sd / qc_mean
        qc_detection = batch_qc.notna().mean(axis=1)
    else:
        qc_cv = pd.Series(np.nan, index=data.index)
        qc_detection = pd.Series(np.nan, index=data.index)
    qc_cv_by_batch[batch] = qc_cv
    qc_detection_by_batch[batch] = qc_detection
qc_cv_matrix = pd.DataFrame(qc_cv_by_batch)
qc_detection_matrix = pd.DataFrame(qc_detection_by_batch)
qc_cv_max = qc_cv_matrix.max(axis=1, skipna=True)
qc_detection_min = qc_detection_matrix.min(axis=1, skipna=True)
qc_batch_coverage_complete = all((qc_count_by_batch[batch] >= 2 for batch in sample_batches))
if not qc_batch_coverage_complete:
    print('\nWarning: some sample batches have fewer than two quantitative QC samples:')
    print(qc_count_by_batch)

def all_batches_pass(cv_threshold):
    if not qc_batch_coverage_complete:
        return pd.Series(False, index=data.index)
    pass_matrix = pd.DataFrame({batch: qc_cv_by_batch[batch].le(cv_threshold) & qc_detection_by_batch[batch].ge(qc_detection_threshold) for batch in sample_batches})
    return pass_matrix.all(axis=1)
qc_pass_30 = all_batches_pass(strict_qc_cv_threshold)
qc_pass_40 = all_batches_pass(rescue_qc_cv_threshold)
blank_ratio_by_batch = {}
if blank_columns:
    blank_all = intensity[blank_columns]
    blank_half_min = blank_all.min(axis=1, skipna=True) / 2
    for batch in sample_batches:
        batch_sample_columns = [column for column in sample_columns if batch_id[column] == batch]
        batch_blank_columns = [column for column in blank_columns if batch_id[column] == batch]
        if not batch_blank_columns:
            batch_blank_columns = blank_columns.copy()
        sample_median = intensity[batch_sample_columns].median(axis=1, skipna=True)
        blank_median = intensity[batch_blank_columns].median(axis=1, skipna=True)
        blank_denominator = blank_median.copy()
        missing_blank = blank_denominator.isna() | blank_denominator.le(0)
        blank_denominator.loc[missing_blank] = blank_half_min.loc[missing_blank]
        ratio = sample_median / blank_denominator
        ratio.loc[blank_denominator.isna() & sample_median.gt(0)] = np.inf
        blank_ratio_by_batch[batch] = ratio
    blank_ratio_matrix = pd.DataFrame(blank_ratio_by_batch)
    sample_blank_ratio_min = blank_ratio_matrix.min(axis=1, skipna=True)
    blank_pass_ratio5 = sample_blank_ratio_min.ge(sample_blank_ratio_threshold)
else:
    print('\nWarning: no Blank columns were identified; blank filtering will provisionally be marked as passed.')
    sample_blank_ratio_min = pd.Series(np.inf, index=data.index)
    blank_pass_ratio5 = pd.Series(True, index=data.index)
peak_pattern = re.compile('\n    (?P<mz>\n        \\d+(?:\\.\\d+)?\n    )\n    \\s*:\\s*\n    (?P<intensity>\n        \\d+(?:\\.\\d+)?\n        (?:[eE][+-]?\\d+)?\n    )\n    ', re.VERBOSE)

def count_valid_msms_peaks(value):
    if pd.isna(value):
        return 0
    text = str(value).strip()
    if not text:
        return 0
    count = 0
    for match in peak_pattern.finditer(text):
        try:
            fragment_mz = float(match.group('mz'))
            fragment_intensity = float(match.group('intensity'))
        except ValueError:
            continue
        if fragment_mz > 0 and fragment_intensity > 0:
            count += 1
    return count
if msms_spectrum_col is not None:
    msms_peak_count = data[msms_spectrum_col].map(count_valid_msms_peaks)
else:
    msms_peak_count = pd.Series(0, index=data.index)
has_valid_msms_spectrum = msms_peak_count >= minimum_msms_peak_count

def text_to_bool(series):
    return series.fillna('').astype(str).str.strip().str.lower().isin({'true', '1', 'yes', 'y'})
if msms_assigned_col is not None:
    msms_assigned = text_to_bool(data[msms_assigned_col])
else:
    msms_assigned = pd.Series(False, index=data.index)
if msms_matched_col is not None:
    msms_matched = text_to_bool(data[msms_matched_col])
else:
    msms_matched = pd.Series(False, index=data.index)

def clean_candidate_name(value):
    text = str(value or '').strip().strip('"')
    text = re.sub('^w/o\\s*MS2:\\s*', '', text, flags=re.I)
    text = re.sub('\\s*\\[IIN-based on:.*?\\]\\s*$', '', text, flags=re.I)
    if text.upper().startswith('NCGC') and '!' in text:
        text = text.split('!', 1)[1]
    text = re.sub('\\s*;\\s*(PlaSMA ID-\\d+|CE\\d+|[A-Z]{14}-[A-Z]{10}-[A-Z])(?:\\s*;.*)?$', '', text, flags=re.I)
    text = re.sub('_major$', '', text, flags=re.I)
    return re.sub('\\s+', ' ', text).strip()
cleaned_name = data[name_col].map(clean_candidate_name)
name_lower = cleaned_name.str.lower()
is_unknown = cleaned_name.eq('') | name_lower.str.contains('unknown|unkown|unidentified|no annotation', regex=True, na=False)
has_specific_candidate_name = ~is_unknown
atomic_mass = {'H': 1.00782503223, 'C': 12.0, 'N': 14.00307400443, 'O': 15.99491461957, 'P': 30.97376199842, 'S': 31.9720711744, 'F': 18.99840316273, 'Cl': 34.968852682, 'Br': 78.9183376, 'I': 126.904468, 'Si': 27.97692653465, 'Na': 22.989769282, 'K': 38.9637064864, 'B': 11.00930536, 'Se': 79.9165218}
formula_pattern = re.compile('([A-Z][a-z]?)(\\d*)')

def calculate_formula_mass(formula):
    if formula is None or pd.isna(formula):
        return np.nan
    text = str(formula).strip()
    if not text:
        return np.nan
    matches = formula_pattern.findall(text)
    if not matches:
        return np.nan
    reconstructed = ''.join((f'{element}{number}' for element, number in matches))
    if reconstructed != text:
        return np.nan
    mass = 0.0
    for element, number in matches:
        if element not in atomic_mass:
            return np.nan
        count = int(number) if number else 1
        mass += atomic_mass[element] * count
    return mass
if formula_col is not None:
    neutral_mass = data[formula_col].map(calculate_formula_mass)
else:
    neutral_mass = pd.Series(np.nan, index=data.index)
has_formula = neutral_mass.notna()
proton_mass = 1.007276466621
electron_mass = 0.000548579909
sodium_ion_mass = 22.989218
potassium_ion_mass = 38.963158
chloride_mass = 34.969401261
bromide_mass = 78.918885
formic_acid_mass = 46.005479303
acetic_acid_mass = 60.021129369
negative_adduct_rules = {'[M-H]-': (1.0, -proton_mass, 1), '[M+Cl]-': (1.0, chloride_mass, 1), '[M+Br]-': (1.0, bromide_mass, 1), '[M+FA-H]-': (1.0, formic_acid_mass - proton_mass, 1), '[M+Hac-H]-': (1.0, acetic_acid_mass - proton_mass, 1), '[M-2H]2-': (1.0, -2 * proton_mass, 2), '[2M-H]-': (2.0, -proton_mass, 1), '[M+Na-2H]-': (1.0, sodium_ion_mass - 2 * proton_mass, 1), '[M+K-2H]-': (1.0, potassium_ion_mass - 2 * proton_mass, 1), '[M]-': (1.0, electron_mass, 1)}
adduct_aliases = {'[M-H]-': '[M-H]-', '[M+CL]-': '[M+Cl]-', '[M+BR]-': '[M+Br]-', '[M+FA-H]-': '[M+FA-H]-', '[M+HCOO]-': '[M+FA-H]-', '[M+FORMATE]-': '[M+FA-H]-', '[M+HAC-H]-': '[M+Hac-H]-', '[M+AC-H]-': '[M+Hac-H]-', '[M+CH3COO]-': '[M+Hac-H]-', '[M+ACETATE]-': '[M+Hac-H]-', '[M-2H]2-': '[M-2H]2-', '[2M-H]-': '[2M-H]-', '[M+NA-2H]-': '[M+Na-2H]-', '[M+K-2H]-': '[M+K-2H]-', '[M]-': '[M]-'}

def canonicalize_negative_adduct(value):
    text = str(value or '').strip()
    text = text.replace(' ', '')
    text = text.replace('−', '-')
    lookup_key = text.upper()
    return adduct_aliases.get(lookup_key, text)
canonical_adduct = data[adduct_col].map(canonicalize_negative_adduct)

def theoretical_mz_from_adduct(molecule_mass, adduct):
    if pd.isna(molecule_mass):
        return np.nan
    if adduct not in negative_adduct_rules:
        return np.nan
    multiplier, shift, charge = negative_adduct_rules[adduct]
    return (multiplier * molecule_mass + shift) / charge
observed_mz = pd.to_numeric(data[mz_col], errors='coerce')
current_theoretical_mz = pd.Series([theoretical_mz_from_adduct(mass, adduct) for mass, adduct in zip(neutral_mass, canonical_adduct)], index=data.index)

def calculate_ppm_error(observed, theoretical):
    result = (observed - theoretical) / theoretical * 1000000.0
    return result.replace([np.inf, -np.inf], np.nan)
current_mz_error_ppm = calculate_ppm_error(observed_mz, current_theoretical_mz)
suggested_adduct_values = []
suggested_ppm_values = []
for mass, observed in zip(neutral_mass, observed_mz):
    if pd.isna(mass) or pd.isna(observed):
        suggested_adduct_values.append(None)
        suggested_ppm_values.append(np.nan)
        continue
    best_adduct = None
    best_ppm = np.nan
    best_absolute_ppm = np.inf
    for adduct in negative_adduct_rules:
        theoretical = theoretical_mz_from_adduct(mass, adduct)
        if pd.isna(theoretical) or theoretical <= 0:
            continue
        ppm = (observed - theoretical) / theoretical * 1000000.0
        absolute_ppm = abs(ppm)
        if absolute_ppm < best_absolute_ppm:
            best_absolute_ppm = absolute_ppm
            best_adduct = adduct
            best_ppm = ppm
    suggested_adduct_values.append(best_adduct)
    suggested_ppm_values.append(best_ppm)
suggested_adduct = pd.Series(suggested_adduct_values, index=data.index)
suggested_mz_error_ppm = pd.Series(suggested_ppm_values, index=data.index)
original_name = data[name_col].fillna('').astype(str)
is_wo_ms2 = original_name.str.contains('^\\s*w/o\\s*MS2', case=False, regex=True)
original_annotation_group = np.select([has_specific_candidate_name & msms_matched, has_specific_candidate_name & msms_assigned, has_specific_candidate_name & is_wo_ms2, has_specific_candidate_name, is_unknown & (msms_assigned | has_valid_msms_spectrum)], ['Named candidate with existing MS/MS match', 'Named candidate with MS/MS assigned', 'w/o MS2 / weak annotation', 'Named candidate without confirmed MS/MS', 'Unknown with MS/MS'], default='Unknown without usable MS/MS')
current_mass_pass_10ppm = current_mz_error_ppm.abs() <= 10
current_mass_pass_20ppm = current_mz_error_ppm.abs() <= 20
alternative_mass_pass_5ppm = suggested_mz_error_ppm.abs() <= 5
alternative_mass_pass_10ppm = suggested_mz_error_ppm.abs() <= 10
strict_mass_consistency = current_mass_pass_10ppm | alternative_mass_pass_5ppm
rescue_mass_consistency = current_mass_pass_20ppm | alternative_mass_pass_10ppm
sample_detection_pass = sample_detection_rate >= sample_detection_threshold
tier_a = qc_pass_30 & blank_pass_ratio5 & sample_detection_pass & has_formula & strict_mass_consistency & has_valid_msms_spectrum & (msms_assigned | msms_matched)
tier_b = ~tier_a & qc_pass_40 & blank_pass_ratio5 & sample_detection_pass & has_formula & rescue_mass_consistency & (has_specific_candidate_name | has_valid_msms_spectrum)
priority_tier = np.select([tier_a, tier_b], ['Tier A: MS/MS + strict QC', 'Tier B: putative/weak annotation + rescue QC'], default='Not retained')
retain_tier_ab = tier_a | tier_b
priority_score = msms_matched.astype(int) * 40 + msms_assigned.astype(int) * 20 + has_valid_msms_spectrum.astype(int) * 10 + (msms_peak_count >= 10).astype(int) * 5 + has_specific_candidate_name.astype(int) * 10 + has_formula.astype(int) * 5 + qc_pass_30.astype(int) * 10 + qc_pass_40.astype(int) * 5 + blank_pass_ratio5.astype(int) * 5 + current_mass_pass_10ppm.astype(int) * 10 + alternative_mass_pass_5ppm.astype(int) * 5 + sample_detection_pass.astype(int) * 5
feature_information_columns = [column for column in data.columns if column not in intensity_columns]
result = data.loc[retain_tier_ab, feature_information_columns].copy()
selected_index = result.index
computed_columns = pd.DataFrame({'Cleaned_candidate': cleaned_name, 'Canonical_negative_adduct': canonical_adduct, 'Estimated_neutral_mass': neutral_mass, 'Current_theoretical_mz': current_theoretical_mz, 'Current_mz_error_ppm': current_mz_error_ppm, 'Suggested_negative_adduct': suggested_adduct, 'Suggested_mz_error_ppm': suggested_mz_error_ppm, 'MSMS_peak_count': msms_peak_count, 'Has_valid_MSMS_spectrum': has_valid_msms_spectrum, 'Has_specific_candidate_name': has_specific_candidate_name, 'Sample_detection_rate': sample_detection_rate, 'QC_detection_rate_min': qc_detection_min, 'QC_CV_max': qc_cv_max, 'QC_pass_all_batches_30pct': qc_pass_30, 'QC_pass_all_batches_40pct': qc_pass_40, 'Sample_blank_ratio_min': sample_blank_ratio_min, 'Blank_pass_all_batches_ratio5': blank_pass_ratio5, 'Original_annotation_group': original_annotation_group, 'Reannotation_priority_score': priority_score, 'Reannotation_priority_tier': priority_tier}, index=data.index)
result = pd.concat([result, computed_columns.loc[selected_index]], axis=1)
tier_sort_order = {'Tier A: MS/MS + strict QC': 0, 'Tier B: putative/weak annotation + rescue QC': 1}
result['_tier_sort'] = result['Reannotation_priority_tier'].map(tier_sort_order).fillna(9)
result = result.sort_values(['_tier_sort', 'Reannotation_priority_score', 'MS/MS matched' if 'MS/MS matched' in result.columns else 'MSMS_peak_count', 'QC_CV_max'], ascending=[True, False, False, True]).drop(columns='_tier_sort').reset_index(drop=True)
result.insert(0, 'Export_rank', np.arange(1, len(result) + 1))
result.to_csv(output_path, index=False, encoding='utf-8-sig', compression='gzip')
output_size_mb = output_path.stat().st_size / 1024 / 1024
split_files = []
if output_size_mb > split_threshold_mb:
    chunks = np.array_split(result, n_split_parts)
    for part_number, chunk in enumerate(chunks, start=1):
        part_path = output_dir / f'BloodNeg_TierAB_with_spectra_part{part_number}.csv.gz'
        chunk.to_csv(part_path, index=False, encoding='utf-8-sig', compression='gzip')
        split_files.append(part_path)
summary = pd.DataFrame({'Metric': ['Input file', 'Raw feature count', 'Clinical sample columns', 'Quantitative QC columns', 'Blank columns', 'Tier A count', 'Tier B count', 'Tier A+B count', 'Specific named candidates', 'Unknown candidates', 'MS/MS assigned', 'MS/MS matched', 'Valid MS/MS spectra', 'QC CV <=30%', 'QC CV 30-40%', 'Output size MB'], 'Value': [str(input_path), len(data), len(sample_columns), len(qc_columns), len(blank_columns), int(tier_a.sum()), int(tier_b.sum()), len(result), int(has_specific_candidate_name.loc[selected_index].sum()), int(is_unknown.loc[selected_index].sum()), int(msms_assigned.loc[selected_index].sum()), int(msms_matched.loc[selected_index].sum()), int(has_valid_msms_spectrum.loc[selected_index].sum()), int(qc_pass_30.loc[selected_index].sum()), int((qc_pass_40.loc[selected_index] & ~qc_pass_30.loc[selected_index]).sum()), round(output_size_mb, 3)]})
summary.to_csv(summary_path, index=False, encoding='utf-8-sig')



print(summary.to_string(index=False))


print('\nSummary files:')

if split_files:
    print('\nThe large file has been split automatically:')
    for file in split_files:
        print(file)
else:
    print('\nThe file is below the splitting threshold; the complete BloodNeg file can be uploaded directly.')


In [ ]:
from pathlib import Path
import pandas as pd
input_path = Path(str(project_path(get_value('science.tables.features_029', 'features_029.txt'))))
preview = pd.read_csv(input_path, sep='\t', header=None, nrows=6, dtype=str, keep_default_na=False, engine='python')



print(preview.iloc[:, :30].to_string(index=False, header=False))



print(preview.iloc[:, -20:].to_string(index=False, header=False))
data = pd.read_csv(input_path, sep='\t', header=4, nrows=5, low_memory=False)
metadata = pd.read_csv(input_path, sep='\t', header=None, nrows=4, dtype=str, keep_default_na=False, engine='python')
metadata = metadata.reindex(columns=range(len(data.columns)), fill_value='')
metadata.columns = data.columns
metadata.index = ['Class', 'File type', 'Injection order', 'Batch ID']



print(len(data.columns))

print(data.columns[:40].tolist())

print(data.columns[-40:].tolist())
for row_name in ['Class', 'File type', 'Batch ID']:
    values = metadata.loc[row_name].fillna('').astype(str).str.strip()
    
    print(f'{row_name}: unique values and counts')
    
    print(values.value_counts(dropna=False).to_string())
file_type = metadata.loc['File type'].fillna('').astype(str).str.strip().str.lower()
batch_id = metadata.loc['Batch ID'].fillna('').astype(str).str.strip()
sample_columns = [col for col in data.columns if file_type.get(col, '') in {'sample', 'biological sample', 'study sample'}]
qc_columns_all = [col for col in data.columns if 'qc' in file_type.get(col, '')]
qc_columns_quantitative = [col for col in qc_columns_all if 'ddms2' not in str(col).lower()]
blank_columns = [col for col in data.columns if 'blank' in file_type.get(col, '')]



print('Number of Sample columns:', len(sample_columns))
print('Number of all QC columns:', len(qc_columns_all))
print('Number of quantitative QC columns:', len(qc_columns_quantitative))
print('Number of Blank columns:', len(blank_columns))
print('\nFirst 10 Sample columns:')
print(sample_columns[:10])
print('\nAll QC columns:')
print(qc_columns_all)
print('\nBlank columns:')
print(blank_columns)
print('\nSample batch distribution:')
print(batch_id.loc[sample_columns].value_counts(dropna=False).to_string())
print('\nQuantitative QC batch distribution:')
print(batch_id.loc[qc_columns_quantitative].value_counts(dropna=False).to_string())


In [ ]:
from pathlib import Path
import math
import re
import numpy as np
import pandas as pd
FLUID = 'Urine'
ION_MODE = 'POS'
INPUT_DIR = Path(str(project_path(get_value('science.paths.directory_098', 'directory_098'))))
MANUAL_INPUT_PATH = None
OUTPUT_DIR = INPUT_DIR / get_value('science.paths.directory_097', 'directory_097')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / get_value('science.tables.result_125', 'result_125.csv.gz')
SUMMARY_PATH = OUTPUT_DIR / get_value('science.tables.metrics_059', 'metrics_059.csv')
SPLIT_PREFIX = 'UrinePos_TierAB_strictQC30_with_spectra_part'
SAMPLE_DETECTION_THRESHOLD = 0.5
QC_DETECTION_THRESHOLD = 0.7
QC_CV_THRESHOLD = 0.3
SAMPLE_BLANK_RATIO_THRESHOLD = 5.0
MINIMUM_MSMS_PEAK_COUNT = 3
SPLIT_THRESHOLD_MB = 20
N_SPLIT_PARTS = 4
if MANUAL_INPUT_PATH is not None:
    input_path = Path(MANUAL_INPUT_PATH)
else:
    height_candidates = list(INPUT_DIR.glob('Height_*.txt'))
    if not height_candidates:
        raise FileNotFoundError(f'No Height_*.txt file was found in:\n{INPUT_DIR}')
    input_path = max(height_candidates, key=lambda p: p.stat().st_size)
if not input_path.exists():
    raise FileNotFoundError(input_path)
print(f'Using {FLUID} {ION_MODE} Height file:')


metadata_raw = pd.read_csv(input_path, sep='\t', header=None, nrows=4, dtype=str, keep_default_na=False, engine='python')
data = pd.read_csv(input_path, sep='\t', header=4, low_memory=False)
metadata_raw = metadata_raw.reindex(columns=range(len(data.columns)), fill_value='')
metadata_raw.columns = data.columns
metadata_raw.index = ['Class', 'File type', 'Injection order', 'Batch ID']
print('\nRaw table shape:', data.shape)

def normalize_column_name(value):
    return re.sub('[^a-z0-9]+', '', str(value).strip().lower())
normalized_column_map = {normalize_column_name(column): column for column in data.columns}

def find_column(candidates, required=True):
    for candidate in candidates:
        key = normalize_column_name(candidate)
        if key in normalized_column_map:
            return normalized_column_map[key]
    if required:
        raise KeyError('Could not find any of these columns:\n' + '\n'.join(candidates))
    return None
alignment_id_col = find_column(['Alignment ID', 'AlignmentID'])
rt_col = find_column(['Average Rt(min)', 'Average RT(min)', 'Average Rt'])
mz_col = find_column(['Average Mz', 'Average m/z', 'AverageMz'])
name_col = find_column(['Metabolite name', 'Metabolite Name'])
adduct_col = find_column(['Adduct type', 'Adduct'])
formula_col = find_column(['Formula'], required=False)
msms_spectrum_col = find_column(['MS/MS spectrum', 'MSMS spectrum'], required=False)
msms_assigned_col = find_column(['MS/MS assigned', 'MSMS assigned'], required=False)
msms_matched_col = find_column(['MS/MS matched', 'MSMS matched'], required=False)
file_type = metadata_raw.loc['File type'].fillna('').astype(str).str.strip().str.lower()
batch_id = metadata_raw.loc['Batch ID'].fillna('').astype(str).str.strip()

def normalize_batch(value):
    value = str(value).strip()
    if not value or value.lower() in {'nan', 'none', 'na'}:
        return 'ALL'
    try:
        numeric = float(value)
        if numeric.is_integer():
            return str(int(numeric))
    except ValueError:
        pass
    return value
batch_id = batch_id.map(normalize_batch)
sample_columns = [column for column in data.columns if file_type.get(column, '') in {'sample', 'biological sample', 'study sample'}]
qc_columns_all = [column for column in data.columns if 'qc' in file_type.get(column, '')]
qc_columns = [column for column in qc_columns_all if 'ddms2' not in str(column).lower()]
blank_columns = [column for column in data.columns if 'blank' in file_type.get(column, '')]
print('\nDetected columns:')
print('Biological samples:', len(sample_columns))
print('All QC columns:', len(qc_columns_all))
print('Quantitative QC columns:', len(qc_columns))
print('Blank columns:', len(blank_columns))
if not sample_columns:
    print('\nUnique File type values:')
    print(sorted(file_type.unique()))
    raise ValueError('No biological sample columns were identified.')
if len(qc_columns) < 2:
    raise ValueError('Fewer than two quantitative QC columns were identified.')
intensity_columns = list(dict.fromkeys(sample_columns + qc_columns + blank_columns))
intensity = data[intensity_columns].apply(pd.to_numeric, errors='coerce')
intensity = intensity.mask(intensity <= 0, np.nan)
sample_detection_rate = intensity[sample_columns].notna().mean(axis=1)
sample_batches = sorted({batch_id[column] for column in sample_columns})
qc_cv_by_batch = {}
qc_detection_by_batch = {}
qc_count_by_batch = {}
for batch in sample_batches:
    batch_qc_columns = [column for column in qc_columns if batch_id[column] == batch]
    if not batch_qc_columns and len(sample_batches) == 1:
        batch_qc_columns = qc_columns.copy()
    qc_count_by_batch[batch] = len(batch_qc_columns)
    if len(batch_qc_columns) >= 2:
        batch_qc = intensity[batch_qc_columns]
        qc_mean = batch_qc.mean(axis=1, skipna=True)
        qc_sd = batch_qc.std(axis=1, skipna=True, ddof=1)
        qc_cv = qc_sd / qc_mean
        qc_detection = batch_qc.notna().mean(axis=1)
    else:
        qc_cv = pd.Series(np.nan, index=data.index)
        qc_detection = pd.Series(np.nan, index=data.index)
    qc_cv_by_batch[batch] = qc_cv
    qc_detection_by_batch[batch] = qc_detection
qc_cv_matrix = pd.DataFrame(qc_cv_by_batch)
qc_detection_matrix = pd.DataFrame(qc_detection_by_batch)
qc_cv_max = qc_cv_matrix.max(axis=1, skipna=True)
qc_detection_min = qc_detection_matrix.min(axis=1, skipna=True)
qc_batch_coverage_complete = all((qc_count_by_batch[batch] >= 2 for batch in sample_batches))
if not qc_batch_coverage_complete:
    print('\nQC counts by batch:')
    print(qc_count_by_batch)
    raise ValueError('At least one sample batch has fewer than two quantitative QC samples.')
qc_pass_30 = pd.DataFrame({batch: qc_cv_by_batch[batch].le(QC_CV_THRESHOLD) & qc_detection_by_batch[batch].ge(QC_DETECTION_THRESHOLD) for batch in sample_batches}).all(axis=1)
if blank_columns:
    blank_all = intensity[blank_columns]
    blank_half_min = blank_all.min(axis=1, skipna=True) / 2
    blank_ratio_by_batch = {}
    for batch in sample_batches:
        batch_sample_columns = [column for column in sample_columns if batch_id[column] == batch]
        batch_blank_columns = [column for column in blank_columns if batch_id[column] == batch]
        if not batch_blank_columns:
            batch_blank_columns = blank_columns.copy()
        sample_median = intensity[batch_sample_columns].median(axis=1, skipna=True)
        blank_median = intensity[batch_blank_columns].median(axis=1, skipna=True)
        blank_denominator = blank_median.copy()
        missing_blank = blank_denominator.isna() | blank_denominator.le(0)
        blank_denominator.loc[missing_blank] = blank_half_min.loc[missing_blank]
        ratio = sample_median / blank_denominator
        ratio.loc[blank_denominator.isna() & sample_median.gt(0)] = np.inf
        blank_ratio_by_batch[batch] = ratio
    blank_ratio_matrix = pd.DataFrame(blank_ratio_by_batch)
    sample_blank_ratio_min = blank_ratio_matrix.min(axis=1, skipna=True)
    blank_pass_ratio5 = sample_blank_ratio_min.ge(SAMPLE_BLANK_RATIO_THRESHOLD)
else:
    print('\nWarning: no Blank columns were identified.')
    sample_blank_ratio_min = pd.Series(np.inf, index=data.index)
    blank_pass_ratio5 = pd.Series(True, index=data.index)
peak_pattern = re.compile('(?P<mz>\\d+(?:\\.\\d+)?)\\s*:\\s*(?P<intensity>\\d+(?:\\.\\d+)?(?:[eE][+-]?\\d+)?)')

def count_valid_msms_peaks(value):
    if pd.isna(value):
        return 0
    text = str(value).strip()
    if not text:
        return 0
    count = 0
    for match in peak_pattern.finditer(text):
        try:
            fragment_mz = float(match.group('mz'))
            fragment_intensity = float(match.group('intensity'))
        except ValueError:
            continue
        if fragment_mz > 0 and fragment_intensity > 0:
            count += 1
    return count
if msms_spectrum_col is not None:
    msms_peak_count = data[msms_spectrum_col].map(count_valid_msms_peaks)
else:
    msms_peak_count = pd.Series(0, index=data.index)
has_valid_msms_spectrum = msms_peak_count.ge(MINIMUM_MSMS_PEAK_COUNT)

def text_to_bool(series):
    return series.fillna('').astype(str).str.strip().str.lower().isin({'true', '1', 'yes', 'y'})
if msms_assigned_col is not None:
    msms_assigned = text_to_bool(data[msms_assigned_col])
else:
    msms_assigned = pd.Series(False, index=data.index)
if msms_matched_col is not None:
    msms_matched = text_to_bool(data[msms_matched_col])
else:
    msms_matched = pd.Series(False, index=data.index)

def clean_candidate_name(value):
    text = str(value or '').strip().strip('"')
    text = re.sub('^w/o\\s*MS2:\\s*', '', text, flags=re.I)
    text = re.sub('\\s*\\[IIN-based on:.*?\\]\\s*$', '', text, flags=re.I)
    if text.upper().startswith('NCGC') and '!' in text:
        text = text.split('!', 1)[1]
    text = re.sub('\\s*;\\s*(PlaSMA ID-\\d+|CE\\d+|[A-Z]{14}-[A-Z]{10}-[A-Z])(?:\\s*;.*)?$', '', text, flags=re.I)
    text = re.sub('_major$', '', text, flags=re.I)
    return re.sub('\\s+', ' ', text).strip()
cleaned_name = data[name_col].map(clean_candidate_name)
name_lower = cleaned_name.str.lower()
is_unknown = cleaned_name.eq('') | name_lower.str.contains('unknown|unkown|unidentified|no annotation', regex=True, na=False)
has_specific_candidate_name = ~is_unknown
atomic_mass = {'H': 1.00782503223, 'C': 12.0, 'N': 14.00307400443, 'O': 15.99491461957, 'P': 30.97376199842, 'S': 31.9720711744, 'F': 18.99840316273, 'Cl': 34.968852682, 'Br': 78.9183376, 'I': 126.904468, 'Si': 27.97692653465, 'Na': 22.989769282, 'K': 38.9637064864, 'B': 11.00930536, 'Se': 79.9165218}
formula_pattern = re.compile('([A-Z][a-z]?)(\\d*)')

def calculate_formula_mass(formula):
    if formula is None or pd.isna(formula):
        return np.nan
    text = str(formula).strip()
    if not text:
        return np.nan
    matches = formula_pattern.findall(text)
    if not matches:
        return np.nan
    reconstructed = ''.join((f'{element}{number}' for element, number in matches))
    if reconstructed != text:
        return np.nan
    mass = 0.0
    for element, number in matches:
        if element not in atomic_mass:
            return np.nan
        count = int(number) if number else 1
        mass += atomic_mass[element] * count
    return mass
if formula_col is not None:
    neutral_mass = data[formula_col].map(calculate_formula_mass)
else:
    neutral_mass = pd.Series(np.nan, index=data.index)
has_formula = neutral_mass.notna()
PROTON = 1.007276466621
ELECTRON = 0.000548579909
SODIUM = 22.989218
POTASSIUM = 38.963158
AMMONIUM = 18.033823
WATER = 18.010564684
ACETONITRILE = 41.026549101
CHLORIDE = 34.969401261
BROMIDE = 78.918885
FORMIC_ACID = 46.005479303
ACETIC_ACID = 60.021129369
ADDUCT_RULES = {'[M+H]+': (1.0, PROTON, 1), '[M+Na]+': (1.0, SODIUM, 1), '[M+K]+': (1.0, POTASSIUM, 1), '[M+NH4]+': (1.0, AMMONIUM, 1), '[M+H-H2O]+': (1.0, PROTON - WATER, 1), '[M+ACN+H]+': (1.0, ACETONITRILE + PROTON, 1), '[M+2H]2+': (1.0, 2 * PROTON, 2), '[M+H+Na]2+': (1.0, PROTON + SODIUM, 2), '[2M+H]+': (2.0, PROTON, 1), '[2M+Na]+': (2.0, SODIUM, 1), '[M]+': (1.0, -ELECTRON, 1)}
ADDUCT_ALIASES = {'[M+H]+': '[M+H]+', '[M+NA]+': '[M+Na]+', '[M+K]+': '[M+K]+', '[M+NH4]+': '[M+NH4]+', '[M+H-H2O]+': '[M+H-H2O]+', '[M+ACN+H]+': '[M+ACN+H]+', '[M+2H]2+': '[M+2H]2+', '[M+H+NA]2+': '[M+H+Na]2+', '[2M+H]+': '[2M+H]+', '[2M+NA]+': '[2M+Na]+', '[M]+': '[M]+'}

def canonicalize_adduct(value):
    text = str(value or '').strip().replace(' ', '').replace('−', '-')
    return ADDUCT_ALIASES.get(text.upper(), text)
canonical_adduct = data[adduct_col].map(canonicalize_adduct)

def theoretical_mz_from_adduct(molecule_mass, adduct):
    if pd.isna(molecule_mass):
        return np.nan
    rule = ADDUCT_RULES.get(adduct)
    if rule is None:
        return np.nan
    multiplier, shift, charge = rule
    return (multiplier * molecule_mass + shift) / charge
observed_mz = pd.to_numeric(data[mz_col], errors='coerce')
current_theoretical_mz = pd.Series([theoretical_mz_from_adduct(mass, adduct) for mass, adduct in zip(neutral_mass, canonical_adduct)], index=data.index)
current_mz_error_ppm = ((observed_mz - current_theoretical_mz) / current_theoretical_mz * 1000000.0).replace([np.inf, -np.inf], np.nan)
suggested_adduct_values = []
suggested_ppm_values = []
for mass, observed in zip(neutral_mass, observed_mz):
    if pd.isna(mass) or pd.isna(observed):
        suggested_adduct_values.append(None)
        suggested_ppm_values.append(np.nan)
        continue
    best_adduct = None
    best_ppm = np.nan
    best_abs_ppm = np.inf
    for adduct in ADDUCT_RULES:
        theoretical = theoretical_mz_from_adduct(mass, adduct)
        if pd.isna(theoretical) or theoretical <= 0:
            continue
        ppm = (observed - theoretical) / theoretical * 1000000.0
        if abs(ppm) < best_abs_ppm:
            best_abs_ppm = abs(ppm)
            best_adduct = adduct
            best_ppm = ppm
    suggested_adduct_values.append(best_adduct)
    suggested_ppm_values.append(best_ppm)
suggested_adduct = pd.Series(suggested_adduct_values, index=data.index)
suggested_mz_error_ppm = pd.Series(suggested_ppm_values, index=data.index)
current_mass_pass_10ppm = current_mz_error_ppm.abs().le(10)
alternative_mass_pass_5ppm = suggested_mz_error_ppm.abs().le(5)
strict_mass_consistency = current_mass_pass_10ppm | alternative_mass_pass_5ppm
sample_detection_pass = sample_detection_rate.ge(SAMPLE_DETECTION_THRESHOLD)
tier_a = qc_pass_30 & blank_pass_ratio5 & sample_detection_pass & has_formula & strict_mass_consistency & has_valid_msms_spectrum & (msms_assigned | msms_matched)
tier_b = ~tier_a & qc_pass_30 & blank_pass_ratio5 & sample_detection_pass & has_formula & strict_mass_consistency & (has_specific_candidate_name | has_valid_msms_spectrum)
retain_tier_ab = tier_a | tier_b
priority_tier = np.select([tier_a, tier_b], ['Tier A: MS/MS + strict QC30', 'Tier B: putative candidate + strict QC30'], default='Not retained')
priority_score = msms_matched.astype(int) * 40 + msms_assigned.astype(int) * 20 + has_valid_msms_spectrum.astype(int) * 10 + (msms_peak_count >= 10).astype(int) * 5 + has_specific_candidate_name.astype(int) * 10 + has_formula.astype(int) * 5 + qc_pass_30.astype(int) * 10 + blank_pass_ratio5.astype(int) * 5 + current_mass_pass_10ppm.astype(int) * 10 + alternative_mass_pass_5ppm.astype(int) * 5 + sample_detection_pass.astype(int) * 5
feature_information_columns = [column for column in data.columns if column not in intensity_columns]
result = data.loc[retain_tier_ab, feature_information_columns].copy()
selected_index = result.index
computed_columns = pd.DataFrame({'Fluid': FLUID, 'Ion_mode': ION_MODE, 'Cleaned_candidate': cleaned_name, 'Canonical_adduct': canonical_adduct, 'Estimated_neutral_mass': neutral_mass, 'Current_theoretical_mz': current_theoretical_mz, 'Current_mz_error_ppm': current_mz_error_ppm, 'Suggested_adduct': suggested_adduct, 'Suggested_mz_error_ppm': suggested_mz_error_ppm, 'MSMS_peak_count': msms_peak_count, 'Has_valid_MSMS_spectrum': has_valid_msms_spectrum, 'Has_specific_candidate_name': has_specific_candidate_name, 'Sample_detection_rate': sample_detection_rate, 'QC_detection_rate_min': qc_detection_min, 'QC_CV_max': qc_cv_max, 'QC_pass_all_batches_30pct': qc_pass_30, 'Sample_blank_ratio_min': sample_blank_ratio_min, 'Blank_pass_all_batches_ratio5': blank_pass_ratio5, 'Reannotation_priority_score': priority_score, 'Reannotation_priority_tier': priority_tier}, index=data.index)
result = pd.concat([result, computed_columns.loc[selected_index]], axis=1)
sort_columns = ['Reannotation_priority_tier', 'Reannotation_priority_score', 'QC_CV_max']
result = result.sort_values(sort_columns, ascending=[True, False, True]).reset_index(drop=True)
result.insert(0, 'Export_rank', np.arange(1, len(result) + 1))
result.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig', compression='gzip')
output_size_mb = OUTPUT_PATH.stat().st_size / 1024 / 1024
split_files = []
if output_size_mb > SPLIT_THRESHOLD_MB:
    chunks = np.array_split(result, N_SPLIT_PARTS)
    for part_number, chunk in enumerate(chunks, start=1):
        part_path = OUTPUT_DIR / f'{SPLIT_PREFIX}{part_number}.csv.gz'
        chunk.to_csv(part_path, index=False, encoding='utf-8-sig', compression='gzip')
        split_files.append(part_path)
summary = pd.DataFrame({'Metric': ['Input file', 'Raw feature count', 'Clinical sample columns', 'Quantitative QC columns', 'Blank columns', 'Sample batch counts', 'Quantitative QC batch counts', 'Tier A count', 'Tier B count', 'Tier A+B strict count', 'Specific named candidates', 'Unknown candidates', 'MS/MS assigned', 'MS/MS matched', 'Valid MS/MS spectra', 'QC CV <=30%', 'Output size MB'], 'Value': [str(input_path), len(data), len(sample_columns), len(qc_columns), len(blank_columns), str(batch_id.loc[sample_columns].value_counts().to_dict()), str(batch_id.loc[qc_columns].value_counts().to_dict()), int(tier_a.sum()), int(tier_b.sum()), len(result), int(has_specific_candidate_name.loc[selected_index].sum()), int(is_unknown.loc[selected_index].sum()), int(msms_assigned.loc[selected_index].sum()), int(msms_matched.loc[selected_index].sum()), int(has_valid_msms_spectrum.loc[selected_index].sum()), int(qc_pass_30.loc[selected_index].sum()), round(output_size_mb, 3)]})
summary.to_csv(SUMMARY_PATH, index=False, encoding='utf-8-sig')



print(summary.to_string(index=False))


print('\nSummary file:')

if split_files:
    
    for file_path in split_files:
        pass
else:
    pass


In [ ]:
from pathlib import Path
import math
import re
import numpy as np
import pandas as pd
FLUID = 'Urine'
ION_MODE = 'NEG'
INPUT_DIR = Path(str(project_path(get_value('science.paths.directory_099', 'directory_099'))))
MANUAL_INPUT_PATH = None
OUTPUT_DIR = INPUT_DIR / get_value('science.paths.directory_097', 'directory_097')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / get_value('science.tables.result_126', 'result_126.csv.gz')
SUMMARY_PATH = OUTPUT_DIR / get_value('science.tables.metrics_060', 'metrics_060.csv')
SPLIT_PREFIX = 'UrineNeg_TierAB_strictQC30_with_spectra_part'
SAMPLE_DETECTION_THRESHOLD = 0.5
QC_DETECTION_THRESHOLD = 0.7
QC_CV_THRESHOLD = 0.3
SAMPLE_BLANK_RATIO_THRESHOLD = 5.0
MINIMUM_MSMS_PEAK_COUNT = 3
SPLIT_THRESHOLD_MB = 20
N_SPLIT_PARTS = 4
if MANUAL_INPUT_PATH is not None:
    input_path = Path(MANUAL_INPUT_PATH)
else:
    height_candidates = list(INPUT_DIR.glob('Height_*.txt'))
    if not height_candidates:
        raise FileNotFoundError(f'No Height_*.txt file was found in:\n{INPUT_DIR}')
    input_path = max(height_candidates, key=lambda p: p.stat().st_size)
if not input_path.exists():
    raise FileNotFoundError(input_path)
print(f'Using {FLUID} {ION_MODE} Height file:')


metadata_raw = pd.read_csv(input_path, sep='\t', header=None, nrows=4, dtype=str, keep_default_na=False, engine='python')
data = pd.read_csv(input_path, sep='\t', header=4, low_memory=False)
metadata_raw = metadata_raw.reindex(columns=range(len(data.columns)), fill_value='')
metadata_raw.columns = data.columns
metadata_raw.index = ['Class', 'File type', 'Injection order', 'Batch ID']
print('\nRaw table shape:', data.shape)

def normalize_column_name(value):
    return re.sub('[^a-z0-9]+', '', str(value).strip().lower())
normalized_column_map = {normalize_column_name(column): column for column in data.columns}

def find_column(candidates, required=True):
    for candidate in candidates:
        key = normalize_column_name(candidate)
        if key in normalized_column_map:
            return normalized_column_map[key]
    if required:
        raise KeyError('Could not find any of these columns:\n' + '\n'.join(candidates))
    return None
alignment_id_col = find_column(['Alignment ID', 'AlignmentID'])
rt_col = find_column(['Average Rt(min)', 'Average RT(min)', 'Average Rt'])
mz_col = find_column(['Average Mz', 'Average m/z', 'AverageMz'])
name_col = find_column(['Metabolite name', 'Metabolite Name'])
adduct_col = find_column(['Adduct type', 'Adduct'])
formula_col = find_column(['Formula'], required=False)
msms_spectrum_col = find_column(['MS/MS spectrum', 'MSMS spectrum'], required=False)
msms_assigned_col = find_column(['MS/MS assigned', 'MSMS assigned'], required=False)
msms_matched_col = find_column(['MS/MS matched', 'MSMS matched'], required=False)
file_type = metadata_raw.loc['File type'].fillna('').astype(str).str.strip().str.lower()
batch_id = metadata_raw.loc['Batch ID'].fillna('').astype(str).str.strip()

def normalize_batch(value):
    value = str(value).strip()
    if not value or value.lower() in {'nan', 'none', 'na'}:
        return 'ALL'
    try:
        numeric = float(value)
        if numeric.is_integer():
            return str(int(numeric))
    except ValueError:
        pass
    return value
batch_id = batch_id.map(normalize_batch)
sample_columns = [column for column in data.columns if file_type.get(column, '') in {'sample', 'biological sample', 'study sample'}]
qc_columns_all = [column for column in data.columns if 'qc' in file_type.get(column, '')]
qc_columns = [column for column in qc_columns_all if 'ddms2' not in str(column).lower()]
blank_columns = [column for column in data.columns if 'blank' in file_type.get(column, '')]
print('\nDetected columns:')
print('Biological samples:', len(sample_columns))
print('All QC columns:', len(qc_columns_all))
print('Quantitative QC columns:', len(qc_columns))
print('Blank columns:', len(blank_columns))
if not sample_columns:
    print('\nUnique File type values:')
    print(sorted(file_type.unique()))
    raise ValueError('No biological sample columns were identified.')
if len(qc_columns) < 2:
    raise ValueError('Fewer than two quantitative QC columns were identified.')
intensity_columns = list(dict.fromkeys(sample_columns + qc_columns + blank_columns))
intensity = data[intensity_columns].apply(pd.to_numeric, errors='coerce')
intensity = intensity.mask(intensity <= 0, np.nan)
sample_detection_rate = intensity[sample_columns].notna().mean(axis=1)
sample_batches = sorted({batch_id[column] for column in sample_columns})
qc_cv_by_batch = {}
qc_detection_by_batch = {}
qc_count_by_batch = {}
for batch in sample_batches:
    batch_qc_columns = [column for column in qc_columns if batch_id[column] == batch]
    if not batch_qc_columns and len(sample_batches) == 1:
        batch_qc_columns = qc_columns.copy()
    qc_count_by_batch[batch] = len(batch_qc_columns)
    if len(batch_qc_columns) >= 2:
        batch_qc = intensity[batch_qc_columns]
        qc_mean = batch_qc.mean(axis=1, skipna=True)
        qc_sd = batch_qc.std(axis=1, skipna=True, ddof=1)
        qc_cv = qc_sd / qc_mean
        qc_detection = batch_qc.notna().mean(axis=1)
    else:
        qc_cv = pd.Series(np.nan, index=data.index)
        qc_detection = pd.Series(np.nan, index=data.index)
    qc_cv_by_batch[batch] = qc_cv
    qc_detection_by_batch[batch] = qc_detection
qc_cv_matrix = pd.DataFrame(qc_cv_by_batch)
qc_detection_matrix = pd.DataFrame(qc_detection_by_batch)
qc_cv_max = qc_cv_matrix.max(axis=1, skipna=True)
qc_detection_min = qc_detection_matrix.min(axis=1, skipna=True)
qc_batch_coverage_complete = all((qc_count_by_batch[batch] >= 2 for batch in sample_batches))
if not qc_batch_coverage_complete:
    print('\nQC counts by batch:')
    print(qc_count_by_batch)
    raise ValueError('At least one sample batch has fewer than two quantitative QC samples.')
qc_pass_30 = pd.DataFrame({batch: qc_cv_by_batch[batch].le(QC_CV_THRESHOLD) & qc_detection_by_batch[batch].ge(QC_DETECTION_THRESHOLD) for batch in sample_batches}).all(axis=1)
if blank_columns:
    blank_all = intensity[blank_columns]
    blank_half_min = blank_all.min(axis=1, skipna=True) / 2
    blank_ratio_by_batch = {}
    for batch in sample_batches:
        batch_sample_columns = [column for column in sample_columns if batch_id[column] == batch]
        batch_blank_columns = [column for column in blank_columns if batch_id[column] == batch]
        if not batch_blank_columns:
            batch_blank_columns = blank_columns.copy()
        sample_median = intensity[batch_sample_columns].median(axis=1, skipna=True)
        blank_median = intensity[batch_blank_columns].median(axis=1, skipna=True)
        blank_denominator = blank_median.copy()
        missing_blank = blank_denominator.isna() | blank_denominator.le(0)
        blank_denominator.loc[missing_blank] = blank_half_min.loc[missing_blank]
        ratio = sample_median / blank_denominator
        ratio.loc[blank_denominator.isna() & sample_median.gt(0)] = np.inf
        blank_ratio_by_batch[batch] = ratio
    blank_ratio_matrix = pd.DataFrame(blank_ratio_by_batch)
    sample_blank_ratio_min = blank_ratio_matrix.min(axis=1, skipna=True)
    blank_pass_ratio5 = sample_blank_ratio_min.ge(SAMPLE_BLANK_RATIO_THRESHOLD)
else:
    print('\nWarning: no Blank columns were identified.')
    sample_blank_ratio_min = pd.Series(np.inf, index=data.index)
    blank_pass_ratio5 = pd.Series(True, index=data.index)
peak_pattern = re.compile('(?P<mz>\\d+(?:\\.\\d+)?)\\s*:\\s*(?P<intensity>\\d+(?:\\.\\d+)?(?:[eE][+-]?\\d+)?)')

def count_valid_msms_peaks(value):
    if pd.isna(value):
        return 0
    text = str(value).strip()
    if not text:
        return 0
    count = 0
    for match in peak_pattern.finditer(text):
        try:
            fragment_mz = float(match.group('mz'))
            fragment_intensity = float(match.group('intensity'))
        except ValueError:
            continue
        if fragment_mz > 0 and fragment_intensity > 0:
            count += 1
    return count
if msms_spectrum_col is not None:
    msms_peak_count = data[msms_spectrum_col].map(count_valid_msms_peaks)
else:
    msms_peak_count = pd.Series(0, index=data.index)
has_valid_msms_spectrum = msms_peak_count.ge(MINIMUM_MSMS_PEAK_COUNT)

def text_to_bool(series):
    return series.fillna('').astype(str).str.strip().str.lower().isin({'true', '1', 'yes', 'y'})
if msms_assigned_col is not None:
    msms_assigned = text_to_bool(data[msms_assigned_col])
else:
    msms_assigned = pd.Series(False, index=data.index)
if msms_matched_col is not None:
    msms_matched = text_to_bool(data[msms_matched_col])
else:
    msms_matched = pd.Series(False, index=data.index)

def clean_candidate_name(value):
    text = str(value or '').strip().strip('"')
    text = re.sub('^w/o\\s*MS2:\\s*', '', text, flags=re.I)
    text = re.sub('\\s*\\[IIN-based on:.*?\\]\\s*$', '', text, flags=re.I)
    if text.upper().startswith('NCGC') and '!' in text:
        text = text.split('!', 1)[1]
    text = re.sub('\\s*;\\s*(PlaSMA ID-\\d+|CE\\d+|[A-Z]{14}-[A-Z]{10}-[A-Z])(?:\\s*;.*)?$', '', text, flags=re.I)
    text = re.sub('_major$', '', text, flags=re.I)
    return re.sub('\\s+', ' ', text).strip()
cleaned_name = data[name_col].map(clean_candidate_name)
name_lower = cleaned_name.str.lower()
is_unknown = cleaned_name.eq('') | name_lower.str.contains('unknown|unkown|unidentified|no annotation', regex=True, na=False)
has_specific_candidate_name = ~is_unknown
atomic_mass = {'H': 1.00782503223, 'C': 12.0, 'N': 14.00307400443, 'O': 15.99491461957, 'P': 30.97376199842, 'S': 31.9720711744, 'F': 18.99840316273, 'Cl': 34.968852682, 'Br': 78.9183376, 'I': 126.904468, 'Si': 27.97692653465, 'Na': 22.989769282, 'K': 38.9637064864, 'B': 11.00930536, 'Se': 79.9165218}
formula_pattern = re.compile('([A-Z][a-z]?)(\\d*)')

def calculate_formula_mass(formula):
    if formula is None or pd.isna(formula):
        return np.nan
    text = str(formula).strip()
    if not text:
        return np.nan
    matches = formula_pattern.findall(text)
    if not matches:
        return np.nan
    reconstructed = ''.join((f'{element}{number}' for element, number in matches))
    if reconstructed != text:
        return np.nan
    mass = 0.0
    for element, number in matches:
        if element not in atomic_mass:
            return np.nan
        count = int(number) if number else 1
        mass += atomic_mass[element] * count
    return mass
if formula_col is not None:
    neutral_mass = data[formula_col].map(calculate_formula_mass)
else:
    neutral_mass = pd.Series(np.nan, index=data.index)
has_formula = neutral_mass.notna()
PROTON = 1.007276466621
ELECTRON = 0.000548579909
SODIUM = 22.989218
POTASSIUM = 38.963158
AMMONIUM = 18.033823
WATER = 18.010564684
ACETONITRILE = 41.026549101
CHLORIDE = 34.969401261
BROMIDE = 78.918885
FORMIC_ACID = 46.005479303
ACETIC_ACID = 60.021129369
ADDUCT_RULES = {'[M-H]-': (1.0, -PROTON, 1), '[M+Cl]-': (1.0, CHLORIDE, 1), '[M+Br]-': (1.0, BROMIDE, 1), '[M+FA-H]-': (1.0, FORMIC_ACID - PROTON, 1), '[M+Hac-H]-': (1.0, ACETIC_ACID - PROTON, 1), '[M-2H]2-': (1.0, -2 * PROTON, 2), '[2M-H]-': (2.0, -PROTON, 1), '[M+Na-2H]-': (1.0, SODIUM - 2 * PROTON, 1), '[M+K-2H]-': (1.0, POTASSIUM - 2 * PROTON, 1), '[M]-': (1.0, ELECTRON, 1)}
ADDUCT_ALIASES = {'[M-H]-': '[M-H]-', '[M+CL]-': '[M+Cl]-', '[M+BR]-': '[M+Br]-', '[M+FA-H]-': '[M+FA-H]-', '[M+HCOO]-': '[M+FA-H]-', '[M+FORMATE]-': '[M+FA-H]-', '[M+HAC-H]-': '[M+Hac-H]-', '[M+AC-H]-': '[M+Hac-H]-', '[M+CH3COO]-': '[M+Hac-H]-', '[M+ACETATE]-': '[M+Hac-H]-', '[M-2H]2-': '[M-2H]2-', '[2M-H]-': '[2M-H]-', '[M+NA-2H]-': '[M+Na-2H]-', '[M+K-2H]-': '[M+K-2H]-', '[M]-': '[M]-'}

def canonicalize_adduct(value):
    text = str(value or '').strip().replace(' ', '').replace('−', '-')
    return ADDUCT_ALIASES.get(text.upper(), text)
canonical_adduct = data[adduct_col].map(canonicalize_adduct)

def theoretical_mz_from_adduct(molecule_mass, adduct):
    if pd.isna(molecule_mass):
        return np.nan
    rule = ADDUCT_RULES.get(adduct)
    if rule is None:
        return np.nan
    multiplier, shift, charge = rule
    return (multiplier * molecule_mass + shift) / charge
observed_mz = pd.to_numeric(data[mz_col], errors='coerce')
current_theoretical_mz = pd.Series([theoretical_mz_from_adduct(mass, adduct) for mass, adduct in zip(neutral_mass, canonical_adduct)], index=data.index)
current_mz_error_ppm = ((observed_mz - current_theoretical_mz) / current_theoretical_mz * 1000000.0).replace([np.inf, -np.inf], np.nan)
suggested_adduct_values = []
suggested_ppm_values = []
for mass, observed in zip(neutral_mass, observed_mz):
    if pd.isna(mass) or pd.isna(observed):
        suggested_adduct_values.append(None)
        suggested_ppm_values.append(np.nan)
        continue
    best_adduct = None
    best_ppm = np.nan
    best_abs_ppm = np.inf
    for adduct in ADDUCT_RULES:
        theoretical = theoretical_mz_from_adduct(mass, adduct)
        if pd.isna(theoretical) or theoretical <= 0:
            continue
        ppm = (observed - theoretical) / theoretical * 1000000.0
        if abs(ppm) < best_abs_ppm:
            best_abs_ppm = abs(ppm)
            best_adduct = adduct
            best_ppm = ppm
    suggested_adduct_values.append(best_adduct)
    suggested_ppm_values.append(best_ppm)
suggested_adduct = pd.Series(suggested_adduct_values, index=data.index)
suggested_mz_error_ppm = pd.Series(suggested_ppm_values, index=data.index)
current_mass_pass_10ppm = current_mz_error_ppm.abs().le(10)
alternative_mass_pass_5ppm = suggested_mz_error_ppm.abs().le(5)
strict_mass_consistency = current_mass_pass_10ppm | alternative_mass_pass_5ppm
sample_detection_pass = sample_detection_rate.ge(SAMPLE_DETECTION_THRESHOLD)
tier_a = qc_pass_30 & blank_pass_ratio5 & sample_detection_pass & has_formula & strict_mass_consistency & has_valid_msms_spectrum & (msms_assigned | msms_matched)
tier_b = ~tier_a & qc_pass_30 & blank_pass_ratio5 & sample_detection_pass & has_formula & strict_mass_consistency & (has_specific_candidate_name | has_valid_msms_spectrum)
retain_tier_ab = tier_a | tier_b
priority_tier = np.select([tier_a, tier_b], ['Tier A: MS/MS + strict QC30', 'Tier B: putative candidate + strict QC30'], default='Not retained')
priority_score = msms_matched.astype(int) * 40 + msms_assigned.astype(int) * 20 + has_valid_msms_spectrum.astype(int) * 10 + (msms_peak_count >= 10).astype(int) * 5 + has_specific_candidate_name.astype(int) * 10 + has_formula.astype(int) * 5 + qc_pass_30.astype(int) * 10 + blank_pass_ratio5.astype(int) * 5 + current_mass_pass_10ppm.astype(int) * 10 + alternative_mass_pass_5ppm.astype(int) * 5 + sample_detection_pass.astype(int) * 5
feature_information_columns = [column for column in data.columns if column not in intensity_columns]
result = data.loc[retain_tier_ab, feature_information_columns].copy()
selected_index = result.index
computed_columns = pd.DataFrame({'Fluid': FLUID, 'Ion_mode': ION_MODE, 'Cleaned_candidate': cleaned_name, 'Canonical_adduct': canonical_adduct, 'Estimated_neutral_mass': neutral_mass, 'Current_theoretical_mz': current_theoretical_mz, 'Current_mz_error_ppm': current_mz_error_ppm, 'Suggested_adduct': suggested_adduct, 'Suggested_mz_error_ppm': suggested_mz_error_ppm, 'MSMS_peak_count': msms_peak_count, 'Has_valid_MSMS_spectrum': has_valid_msms_spectrum, 'Has_specific_candidate_name': has_specific_candidate_name, 'Sample_detection_rate': sample_detection_rate, 'QC_detection_rate_min': qc_detection_min, 'QC_CV_max': qc_cv_max, 'QC_pass_all_batches_30pct': qc_pass_30, 'Sample_blank_ratio_min': sample_blank_ratio_min, 'Blank_pass_all_batches_ratio5': blank_pass_ratio5, 'Reannotation_priority_score': priority_score, 'Reannotation_priority_tier': priority_tier}, index=data.index)
result = pd.concat([result, computed_columns.loc[selected_index]], axis=1)
sort_columns = ['Reannotation_priority_tier', 'Reannotation_priority_score', 'QC_CV_max']
result = result.sort_values(sort_columns, ascending=[True, False, True]).reset_index(drop=True)
result.insert(0, 'Export_rank', np.arange(1, len(result) + 1))
result.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig', compression='gzip')
output_size_mb = OUTPUT_PATH.stat().st_size / 1024 / 1024
split_files = []
if output_size_mb > SPLIT_THRESHOLD_MB:
    chunks = np.array_split(result, N_SPLIT_PARTS)
    for part_number, chunk in enumerate(chunks, start=1):
        part_path = OUTPUT_DIR / f'{SPLIT_PREFIX}{part_number}.csv.gz'
        chunk.to_csv(part_path, index=False, encoding='utf-8-sig', compression='gzip')
        split_files.append(part_path)
summary = pd.DataFrame({'Metric': ['Input file', 'Raw feature count', 'Clinical sample columns', 'Quantitative QC columns', 'Blank columns', 'Sample batch counts', 'Quantitative QC batch counts', 'Tier A count', 'Tier B count', 'Tier A+B strict count', 'Specific named candidates', 'Unknown candidates', 'MS/MS assigned', 'MS/MS matched', 'Valid MS/MS spectra', 'QC CV <=30%', 'Output size MB'], 'Value': [str(input_path), len(data), len(sample_columns), len(qc_columns), len(blank_columns), str(batch_id.loc[sample_columns].value_counts().to_dict()), str(batch_id.loc[qc_columns].value_counts().to_dict()), int(tier_a.sum()), int(tier_b.sum()), len(result), int(has_specific_candidate_name.loc[selected_index].sum()), int(is_unknown.loc[selected_index].sum()), int(msms_assigned.loc[selected_index].sum()), int(msms_matched.loc[selected_index].sum()), int(has_valid_msms_spectrum.loc[selected_index].sum()), int(qc_pass_30.loc[selected_index].sum()), round(output_size_mb, 3)]})
summary.to_csv(SUMMARY_PATH, index=False, encoding='utf-8-sig')



print(summary.to_string(index=False))


print('\nSummary file:')

if split_files:
    
    for file_path in split_files:
        pass
else:
    pass


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
DICTIONARY_XLSX = Path(str(project_path(get_value('science.tables.metrics_061', 'metrics_061.xlsx'))))
HEIGHT_FILES = {'Blood': {'POS': Path(str(project_path(get_value('science.tables.features_028', 'features_028.txt')))), 'NEG': Path(str(project_path(get_value('science.tables.features_030', 'features_030.txt'))))}, 'Urine': {'POS': Path(str(project_path(get_value('science.tables.features_031', 'features_031.txt')))), 'NEG': Path(str(project_path(get_value('science.tables.features_029', 'features_029.txt'))))}}
OUTPUT_DIR = DICTIONARY_XLSX.parent / get_value('science.paths.directory_067', 'directory_067')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RT_CHECK_TOLERANCE_MIN = 0.01
MZ_CHECK_TOLERANCE_PPM = 5.0
READ_CHUNK_SIZE = 5000

def normalize_alignment_id(value):
    text = str(value).strip()
    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
    except Exception:
        pass
    return text

def canonical_sample_id(column_name):
    text = str(column_name).strip()
    match = re.search('(FZTM\\d+-\\d+[A-Za-z])$', text, flags=re.I)
    if match:
        return match.group(1).upper()
    text = re.sub('_(CP|CN)(?=\\d|_)', '_C', text, flags=re.I)
    return text

def ppm_difference(observed, expected):
    if expected == 0:
        return np.inf
    return abs(observed - expected) / abs(expected) * 1000000.0

def check_input_file(path, label):
    if path.exists():
        
        return
    message = [f'[ERROR] {label} was not found:', str(path)]
    if path.parent.exists():
        candidates = sorted(path.parent.glob('Height_*.txt'))
        if candidates:
            message.append('\nHeight files in this directory:')
            for candidate in candidates:
                size_mb = candidate.stat().st_size / 1024 / 1024
                message.append(f'  - {candidate.name} ({size_mb:.2f} MB)')
    raise FileNotFoundError('\n'.join(message))

def make_unique_feature_names(values):
    used = set()
    next_suffix = {}
    result = []
    for value in values:
        base = str(value).strip()
        if not base or base.lower() in {'nan', 'none'}:
            base = 'Unnamed_feature'
        if base not in used:
            candidate = base
            used.add(candidate)
            next_suffix.setdefault(base, 1)
            result.append(candidate)
            continue
        suffix = next_suffix.get(base, 1)
        while f'{base}.{suffix}' in used:
            suffix += 1
        candidate = f'{base}.{suffix}'
        used.add(candidate)
        next_suffix[base] = suffix + 1
        result.append(candidate)
    return result

def read_feature_dictionary(sheet_name):
    dictionary = pd.read_excel(DICTIONARY_XLSX, sheet_name=sheet_name, dtype={'Feature ID': str, 'Alignment ID': str})
    required_columns = {'Feature ID', 'Ion mode', 'Alignment ID', 'RT (min)', 'Observed m/z', 'Cleaned candidate'}
    missing_columns = required_columns.difference(dictionary.columns)
    if missing_columns:
        raise KeyError(f'{sheet_name} is missing required columns: {sorted(missing_columns)}')
    dictionary['Feature ID'] = dictionary['Feature ID'].astype(str).str.strip()
    dictionary['Ion mode'] = dictionary['Ion mode'].astype(str).str.strip().str.upper()
    dictionary['Alignment ID'] = dictionary['Alignment ID'].map(normalize_alignment_id)
    if dictionary['Feature ID'].duplicated().any():
        duplicates = dictionary.loc[dictionary['Feature ID'].duplicated(keep=False), 'Feature ID']
        raise ValueError(f'{sheet_name} has nonunique Feature IDs:\n{duplicates.head(20).to_list()}')
    dictionary['Matrix feature name'] = make_unique_feature_names(dictionary['Cleaned candidate'].tolist())
    if dictionary['Matrix feature name'].duplicated().any():
        raise ValueError(f'{sheet_name} still has duplicate generated matrix feature names.')
    invalid_modes = sorted(set(dictionary['Ion mode']) - {'POS', 'NEG'})
    if invalid_modes:
        raise ValueError(f'{sheet_name} has invalid Ion mode values: {invalid_modes}')
    return dictionary

def read_height_layout(height_path):
    header = pd.read_csv(height_path, sep='\t', header=4, nrows=0, engine='python')
    metadata = pd.read_csv(height_path, sep='\t', header=None, nrows=4, dtype=str, keep_default_na=False, engine='python')
    metadata = metadata.reindex(columns=range(len(header.columns)), fill_value='')
    metadata.columns = header.columns
    metadata.index = ['Class', 'File type', 'Injection order', 'Batch ID']
    file_type = metadata.loc['File type'].fillna('').astype(str).str.strip().str.lower()
    sample_columns = [column for column in header.columns if file_type.get(column, '') in {'sample', 'biological sample', 'study sample'}]
    if not sample_columns:
        raise ValueError(f'No Sample columns were identified in {height_path.name}.')
    sample_ids = [canonical_sample_id(column) for column in sample_columns]
    duplicate_ids = pd.Series(sample_ids).value_counts()
    duplicate_ids = duplicate_ids[duplicate_ids > 1]
    if not duplicate_ids.empty:
        raise ValueError(f'{height_path.name} has nonunique standardized sample IDs:\n{duplicate_ids.head(20)}')
    sample_metadata = pd.DataFrame({'Sample ID': sample_ids, 'Raw column name': sample_columns, 'Class': [metadata.loc['Class', column] for column in sample_columns], 'Batch ID': [metadata.loc['Batch ID', column] for column in sample_columns], 'Injection order': [metadata.loc['Injection order', column] for column in sample_columns]})
    return {'all_columns': list(header.columns), 'sample_columns': sample_columns, 'sample_ids': sample_ids, 'sample_metadata': sample_metadata}

def extract_mode_matrix(height_path, feature_rows, modality, ion_mode):
    layout = read_height_layout(height_path)
    expected_ids = feature_rows['Alignment ID'].map(normalize_alignment_id).tolist()
    if len(expected_ids) != len(set(expected_ids)):
        duplicates = pd.Series(expected_ids).value_counts()
        duplicates = duplicates[duplicates > 1]
        raise ValueError(f'{modality} {ion_mode} has nonunique Alignment IDs in the feature dictionary:\n{duplicates.head(20)}')
    target_ids = set(expected_ids)
    information_columns = ['Alignment ID', 'Average Rt(min)', 'Average Mz']
    missing_information_columns = [column for column in information_columns if column not in layout['all_columns']]
    if missing_information_columns:
        raise KeyError(f'{height_path.name} is missing information columns: {missing_information_columns}')
    use_columns = information_columns + layout['sample_columns']
    selected_chunks = []
    for chunk in pd.read_csv(height_path, sep='\t', header=4, usecols=use_columns, chunksize=READ_CHUNK_SIZE, low_memory=False):
        chunk['_Alignment_ID_normalized'] = chunk['Alignment ID'].map(normalize_alignment_id)
        selected = chunk[chunk['_Alignment_ID_normalized'].isin(target_ids)].copy()
        if not selected.empty:
            selected_chunks.append(selected)
    if not selected_chunks:
        raise ValueError(f'No target features were found in {height_path.name}.')
    selected_data = pd.concat(selected_chunks, axis=0, ignore_index=True)
    repeated_height_ids = selected_data['_Alignment_ID_normalized'].value_counts()
    repeated_height_ids = repeated_height_ids[repeated_height_ids > 1]
    if not repeated_height_ids.empty:
        raise ValueError(f'{height_path.name} has multiple rows for the same Alignment ID:\n{repeated_height_ids.head(20)}')
    selected_data = selected_data.set_index('_Alignment_ID_normalized', drop=False)
    missing_ids = [alignment_id for alignment_id in expected_ids if alignment_id not in selected_data.index]
    if missing_ids:
        raise ValueError(f'{modality} {ion_mode}: {len(missing_ids)} Alignment IDs were not found.\nFirst 20: {missing_ids[:20]}')
    selected_data = selected_data.loc[expected_ids]
    expected_rt = pd.to_numeric(feature_rows['RT (min)'], errors='coerce').to_numpy()
    observed_rt = pd.to_numeric(selected_data['Average Rt(min)'], errors='coerce').to_numpy()
    expected_mz = pd.to_numeric(feature_rows['Observed m/z'], errors='coerce').to_numpy()
    observed_mz = pd.to_numeric(selected_data['Average Mz'], errors='coerce').to_numpy()
    rt_difference = np.abs(observed_rt - expected_rt)
    mz_ppm_difference = np.array([ppm_difference(observed, expected) if np.isfinite(observed) and np.isfinite(expected) else np.nan for observed, expected in zip(observed_mz, expected_mz)])
    bad_rt = np.where(np.isfinite(rt_difference) & (rt_difference > RT_CHECK_TOLERANCE_MIN))[0]
    bad_mz = np.where(np.isfinite(mz_ppm_difference) & (mz_ppm_difference > MZ_CHECK_TOLERANCE_PPM))[0]
    if len(bad_rt) or len(bad_mz):
        problem_indices = sorted(set(bad_rt.tolist()) | set(bad_mz.tolist()))[:20]
        problem_rows = feature_rows.iloc[problem_indices][['Feature ID', 'Alignment ID', 'RT (min)', 'Observed m/z', 'Cleaned candidate']].copy()
        problem_rows['Height RT'] = observed_rt[problem_indices]
        problem_rows['Height m/z'] = observed_mz[problem_indices]
        problem_rows['RT difference'] = rt_difference[problem_indices]
        problem_rows['m/z difference ppm'] = mz_ppm_difference[problem_indices]
        raise ValueError(f'{modality} {ion_mode}: the feature dictionary does not fully match the Height file; the wrong Height version may have been selected.\n\n{problem_rows.to_string(index=False)}')
    intensity = selected_data[layout['sample_columns']].apply(pd.to_numeric, errors='coerce').transpose()
    intensity.index = layout['sample_ids']
    intensity.index.name = 'Sample ID'
    intensity.columns = feature_rows['Matrix feature name'].astype(str).tolist()
    sample_metadata = layout['sample_metadata'].set_index('Sample ID')
    sample_metadata.columns = [f'{ion_mode} {column}' for column in sample_metadata.columns]
    validation = {'Modality': modality, 'Ion mode': ion_mode, 'Height file': str(height_path), 'Height sample count': len(layout['sample_columns']), 'Requested feature count': len(feature_rows), 'Matched feature count': len(selected_data), 'Maximum RT difference (min)': float(np.nanmax(rt_difference)) if np.isfinite(rt_difference).any() else np.nan, 'Maximum m/z difference (ppm)': float(np.nanmax(mz_ppm_difference)) if np.isfinite(mz_ppm_difference).any() else np.nan}
    return (intensity, sample_metadata, validation)

def build_modality_matrix(modality, dictionary):
    mode_matrices = {}
    mode_metadata = {}
    validations = []
    for ion_mode in ['POS', 'NEG']:
        mode_features = dictionary.loc[dictionary['Ion mode'] == ion_mode].copy()
        if mode_features.empty:
            print(f'[WARNING] {modality} has no {ion_mode} features.')
            continue
        
        matrix, metadata, validation = extract_mode_matrix(HEIGHT_FILES[modality][ion_mode], mode_features, modality, ion_mode)
        mode_matrices[ion_mode] = matrix
        mode_metadata[ion_mode] = metadata
        validations.append(validation)
    if not mode_matrices:
        raise ValueError(f'{modality} has no POS or NEG features available for extraction.')
    modes = list(mode_matrices)
    reference_mode = modes[0]
    reference_ids = list(mode_matrices[reference_mode].index)
    reference_set = set(reference_ids)
    for ion_mode in modes[1:]:
        current_set = set(mode_matrices[ion_mode].index)
        if current_set != reference_set:
            only_reference = sorted(reference_set - current_set)
            only_current = sorted(current_set - reference_set)
            raise ValueError(f'{modality} POS/NEG samples differ.\nOnly in {reference_mode}: {only_reference[:20]}\nOnly in {ion_mode}: {only_current[:20]}')
        mode_matrices[ion_mode] = mode_matrices[ion_mode].reindex(reference_ids)
        mode_metadata[ion_mode] = mode_metadata[ion_mode].reindex(reference_ids)
    combined_matrix = pd.concat([mode_matrices[ion_mode] for ion_mode in modes], axis=1)
    feature_order = dictionary['Matrix feature name'].astype(str).tolist()
    combined_matrix = combined_matrix[feature_order]
    if combined_matrix.columns.duplicated().any():
        duplicate_columns = combined_matrix.columns[combined_matrix.columns.duplicated(keep=False)].tolist()
        raise ValueError(f'{modality} still has duplicate final feature column names: {duplicate_columns[:20]}')
    combined_metadata = pd.concat([mode_metadata[ion_mode] for ion_mode in modes], axis=1)
    matrix_path = OUTPUT_DIR / f"{modality}{get_value('science.tables.features_032', 'features_032.csv.gz')}"
    metadata_path = OUTPUT_DIR / f"{modality}{get_value('science.tables.result_127', 'result_127.csv')}"
    dictionary_output_path = OUTPUT_DIR / f"{modality}{get_value('science.tables.result_128', 'result_128.csv.gz')}"
    combined_matrix.to_csv(matrix_path, compression='gzip', encoding='utf-8-sig')
    combined_metadata.to_csv(metadata_path, encoding='utf-8-sig')
    dictionary.to_csv(dictionary_output_path, compression='gzip', encoding='utf-8-sig', index=False)
    final_summary = {'Modality': modality, 'Final sample count': len(combined_matrix), 'Final feature count': len(combined_matrix.columns), 'Duplicate matrix column names': int(combined_matrix.columns.duplicated().sum()), 'Missing intensity cells': int(combined_matrix.isna().sum().sum()), 'Zero intensity cells': int((combined_matrix == 0).sum().sum()), 'Output matrix': str(matrix_path)}
    return (combined_matrix, combined_metadata, validations, final_summary)

print('Input file checks')

check_input_file(DICTIONARY_XLSX, 'Final feature dictionary workbook')
for modality in ['Blood', 'Urine']:
    for ion_mode in ['POS', 'NEG']:
        check_input_file(HEIGHT_FILES[modality][ion_mode], f'{modality} {ion_mode} Height file')
blood_dictionary = read_feature_dictionary(get_value('science.translation.preprocess_001', 'Blood final dictionary'))
urine_dictionary = read_feature_dictionary(get_value('science.translation.preprocess_002', 'Urine final dictionary'))

print(f"Blood: {len(blood_dictionary)} features; POS={(blood_dictionary['Ion mode'] == 'POS').sum()}, NEG={(blood_dictionary['Ion mode'] == 'NEG').sum()}")
print(f"Urine: {len(urine_dictionary)} features; POS={(urine_dictionary['Ion mode'] == 'POS').sum()}, NEG={(urine_dictionary['Ion mode'] == 'NEG').sum()}")
blood_matrix, blood_sample_metadata, blood_validations, blood_summary = build_modality_matrix('Blood', blood_dictionary)
urine_matrix, urine_sample_metadata, urine_validations, urine_summary = build_modality_matrix('Urine', urine_dictionary)
validation_table = pd.DataFrame(blood_validations + urine_validations)
summary_table = pd.DataFrame([blood_summary, urine_summary])
validation_path = OUTPUT_DIR / get_value('science.tables.features_033', 'features_033.csv')
summary_path = OUTPUT_DIR / get_value('science.tables.features_034', 'features_034.csv')
validation_table.to_csv(validation_path, index=False, encoding='utf-8-sig')
summary_table.to_csv(summary_path, index=False, encoding='utf-8-sig')



print('\nVerification results by ionization mode:')

print('\nFinal matrix summary:')

print('\nFirst 5 rows and 8 columns of the blood matrix:')
display(blood_matrix.iloc[:5, :8])
print('\nFirst 5 rows and 8 columns of the urine matrix:')
display(urine_matrix.iloc[:5, :8])



for path in sorted(OUTPUT_DIR.iterdir()):
    pass


In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print
warnings.filterwarnings('ignore', category=RuntimeWarning)
BASE_DIR = Path(str(project_path(get_value('science.paths.directory_100', 'directory_100'))))
BLOOD_MATRIX_CANDIDATES = [BASE_DIR / get_value('science.tables.features_035', 'features_035.csv') / get_value('science.tables.features_035', 'features_035.csv'), BASE_DIR / get_value('science.tables.features_035', 'features_035.csv')]
URINE_MATRIX_CANDIDATES = [BASE_DIR / get_value('science.tables.features_036', 'features_036.csv') / get_value('science.tables.features_036', 'features_036.csv'), BASE_DIR / get_value('science.tables.features_036', 'features_036.csv')]
BLOOD_METADATA_PATH = BASE_DIR / get_value('science.tables.result_129', 'result_129.csv')
URINE_METADATA_PATH = BASE_DIR / get_value('science.tables.result_130', 'result_130.csv')
OUTPUT_DIR = BASE_DIR / get_value('science.paths.directory_068', 'directory_068')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
BLOOD_MAP_PATH = OUTPUT_DIR / get_value('science.tables.result_131', 'result_131.csv')
URINE_MAP_PATH = OUTPUT_DIR / get_value('science.tables.result_132', 'result_132.csv')

def resolve_existing_path(candidates, label):
    for path in candidates:
        if path.exists() and path.is_file():
            
            return path
    attempted = '\n'.join((str(path) for path in candidates))
    raise FileNotFoundError(f'The {label} file was not found. Paths checked:\n{attempted}')

def clean_sample_id(value):
    if pd.isna(value):
        return ''
    text = str(value)
    text = text.replace('\u3000', ' ')
    text = text.strip().strip('"').strip("'").strip()
    if re.fullmatch('\\d+\\.0', text):
        text = text[:-2]
    return text.upper()

def clean_feature_name(value):
    text = str(value)
    text = text.replace('\u3000', ' ')
    text = text.strip().strip('"').strip("'").strip()
    text = re.sub('\\s+', ' ', text)
    return text

def canonical_original_sample_id(value):
    text = clean_sample_id(value)
    match = re.search('(FZTM\\d+-\\d+[A-Z])$', text, flags=re.I)
    if match:
        return match.group(1).upper()
    return text

def read_matrix(path, modality):
    matrix = pd.read_csv(path, index_col=0, low_memory=False)
    matrix.index = [clean_sample_id(value) for value in matrix.index]
    matrix.columns = [clean_feature_name(value) for value in matrix.columns]
    if '' in matrix.index:
        raise ValueError(f'The {modality} matrix contains empty sample names.')
    if matrix.index.duplicated().any():
        duplicates = pd.Series(matrix.index).value_counts()
        duplicates = duplicates[duplicates > 1]
        display(duplicates.head(30))
        raise ValueError(f'The {modality} matrix contains duplicate sample names.')
    if matrix.columns.duplicated().any():
        duplicates = pd.Series(matrix.columns).value_counts()
        duplicates = duplicates[duplicates > 1]
        display(duplicates.head(30))
        raise ValueError(f'The {modality} matrix contains duplicate feature column names.')
    matrix = matrix.apply(pd.to_numeric, errors='coerce')
    if np.isinf(matrix.to_numpy()).any():
        raise ValueError(f'The {modality} matrix contains positive or negative infinity.')
    return matrix

def infer_cohort_mapping(blood_matrix, urine_matrix):
    blood_ids = set(blood_matrix.index)
    urine_ids = set(urine_matrix.index)
    internal_ids = sorted(blood_ids.intersection(urine_ids))
    blood_external_ids = sorted(blood_ids - urine_ids)
    urine_external_ids = sorted(urine_ids - blood_ids)
    all_ids = sorted(blood_ids.union(urine_ids))
    rows = []
    for sample_id in all_ids:
        in_blood = sample_id in blood_ids
        in_urine = sample_id in urine_ids
        if in_blood and in_urine:
            cohort = 'Internal'
            center = 'SCH'
        elif in_blood:
            cohort = 'Blood_external'
            center = 'QLH'
        else:
            cohort = 'Urine_external'
            center = 'QLH'
        rows.append({'Sample ID': sample_id, 'Blood available': int(in_blood), 'Urine available': int(in_urine), 'Center': center, 'Cohort': cohort})
    mapping = pd.DataFrame(rows)
    counts = {'Internal': len(internal_ids), 'Blood_external': len(blood_external_ids), 'Urine_external': len(urine_external_ids)}
    return (mapping, counts)

def sample_qc_table(matrix_na, cohort_mapping, modality):
    detected_count = matrix_na.notna().sum(axis=1)
    missing_count = matrix_na.isna().sum(axis=1)
    missing_rate = matrix_na.isna().mean(axis=1)
    total_intensity = matrix_na.sum(axis=1, skipna=True)
    median_intensity = matrix_na.median(axis=1, skipna=True)
    mean_intensity = matrix_na.mean(axis=1, skipna=True)
    result = pd.DataFrame({'Sample ID': matrix_na.index, 'Modality': modality, 'Detected feature count': detected_count.values, 'Missing feature count': missing_count.values, 'Missing rate': missing_rate.values, 'Total raw intensity': total_intensity.values, 'Median detected intensity': median_intensity.values, 'Mean detected intensity': mean_intensity.values, 'Log10 total raw intensity': np.log10(total_intensity.values + 1)})
    result = result.merge(cohort_mapping[['Sample ID', 'Center', 'Cohort']], on='Sample ID', how='left', validate='one_to_one')
    return result

def feature_qc_table(matrix_na, cohort_mapping, modality):
    cohort_series = cohort_mapping.set_index('Sample ID').loc[matrix_na.index, 'Cohort']
    result = pd.DataFrame({'Feature': matrix_na.columns, 'Modality': modality, 'Overall detection rate': matrix_na.notna().mean(axis=0).values, 'Overall missing rate': matrix_na.isna().mean(axis=0).values, 'Overall median detected intensity': matrix_na.median(axis=0, skipna=True).values})
    relevant_cohorts = ['Internal', 'Blood_external'] if modality == 'Blood' else ['Internal', 'Urine_external']
    for cohort in relevant_cohorts:
        cohort_ids = cohort_series.index[cohort_series == cohort]
        submatrix = matrix_na.loc[cohort_ids]
        result[f'{cohort} sample count'] = len(submatrix)
        result[f'{cohort} detection rate'] = submatrix.notna().mean(axis=0).values
        result[f'{cohort} missing rate'] = submatrix.isna().mean(axis=0).values
        result[f'{cohort} median detected intensity'] = submatrix.median(axis=0, skipna=True).values
    result['Internal detection >= 50%'] = result['Internal detection rate'] >= 0.5
    return result

def plot_histogram(values, title, xlabel, output_path):
    figure = plt.figure(figsize=(7, 5))
    plt.hist(pd.Series(values).dropna(), bins=30)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel('Sample count')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(figure)

def plot_scatter(x, y, title, xlabel, ylabel, output_path):
    figure = plt.figure(figsize=(7, 5))
    plt.scatter(x, y, s=35, alpha=0.75)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(figure)

print('Stage 1: read and audit the original Height matrices')

BLOOD_MATRIX_PATH = resolve_existing_path(BLOOD_MATRIX_CANDIDATES, 'Blood matrix')
URINE_MATRIX_PATH = resolve_existing_path(URINE_MATRIX_CANDIDATES, 'Urine matrix')
blood = read_matrix(BLOOD_MATRIX_PATH, 'Blood')
urine = read_matrix(URINE_MATRIX_PATH, 'Urine')
print(f'\nBlood: {blood.shape[0]} samples × {blood.shape[1]} features')
print(f'Urine: {urine.shape[0]} samples × {urine.shape[1]} features')
cohort_mapping, cohort_counts = infer_cohort_mapping(blood, urine)
print('\nCohort identification results:')
for key, value in cohort_counts.items():
    pass
EXPECTED_COUNTS = {'Internal': 109, 'Blood_external': 72, 'Urine_external': 33}
if cohort_counts != EXPECTED_COUNTS:
    raise ValueError(f'Cohort counts differ from the expected values.\nObserved: {cohort_counts}\nExpected: {EXPECTED_COUNTS}\nCheck the sample-name format and mappings.')
cohort_mapping_path = OUTPUT_DIR / get_value('science.tables.result_133', 'result_133.csv')
cohort_mapping.to_csv(cohort_mapping_path, index=False, encoding='utf-8-sig')
blood.to_csv(OUTPUT_DIR / get_value('science.tables.features_037', 'features_037.csv.gz'), compression='gzip', encoding='utf-8-sig')
urine.to_csv(OUTPUT_DIR / get_value('science.tables.features_038', 'features_038.csv.gz'), compression='gzip', encoding='utf-8-sig')
blood_na = blood.mask(blood == 0)
urine_na = urine.mask(urine == 0)
blood_na.to_csv(OUTPUT_DIR / get_value('science.tables.result_134', 'result_134.csv.gz'), compression='gzip', encoding='utf-8-sig')
urine_na.to_csv(OUTPUT_DIR / get_value('science.tables.result_135', 'result_135.csv.gz'), compression='gzip', encoding='utf-8-sig')
blood_sample_qc = sample_qc_table(blood_na, cohort_mapping, 'Blood')
urine_sample_qc = sample_qc_table(urine_na, cohort_mapping, 'Urine')
blood_feature_qc = feature_qc_table(blood_na, cohort_mapping, 'Blood')
urine_feature_qc = feature_qc_table(urine_na, cohort_mapping, 'Urine')
blood_sample_qc.to_csv(OUTPUT_DIR / get_value('science.tables.result_136', 'result_136.csv'), index=False, encoding='utf-8-sig')
urine_sample_qc.to_csv(OUTPUT_DIR / get_value('science.tables.result_137', 'result_137.csv'), index=False, encoding='utf-8-sig')
blood_feature_qc.to_csv(OUTPUT_DIR / get_value('science.tables.result_138', 'result_138.csv'), index=False, encoding='utf-8-sig')
urine_feature_qc.to_csv(OUTPUT_DIR / get_value('science.tables.result_139', 'result_139.csv'), index=False, encoding='utf-8-sig')
plot_histogram(blood_sample_qc['Missing rate'], 'Blood sample missing-rate distribution', 'Missing rate', FIGURE_DIR / get_value('science.figures.plot_092', 'plot_092.png'))
plot_histogram(urine_sample_qc['Missing rate'], 'Urine sample missing-rate distribution', 'Missing rate', FIGURE_DIR / get_value('science.figures.plot_093', 'plot_093.png'))
plot_histogram(blood_sample_qc['Log10 total raw intensity'], 'Blood total raw intensity distribution', 'log10(total raw intensity + 1)', FIGURE_DIR / get_value('science.figures.plot_094', 'plot_094.png'))
plot_histogram(urine_sample_qc['Log10 total raw intensity'], 'Urine total raw intensity distribution', 'log10(total raw intensity + 1)', FIGURE_DIR / get_value('science.figures.plot_095', 'plot_095.png'))
plot_scatter(blood_sample_qc['Missing rate'], blood_sample_qc['Log10 total raw intensity'], 'Blood missing rate vs total intensity', 'Missing rate', 'log10(total raw intensity + 1)', FIGURE_DIR / get_value('science.figures.plot_096', 'plot_096.png'))
plot_scatter(urine_sample_qc['Missing rate'], urine_sample_qc['Log10 total raw intensity'], 'Urine missing rate vs total intensity', 'Missing rate', 'log10(total raw intensity + 1)', FIGURE_DIR / get_value('science.figures.plot_097', 'plot_097.png'))

def read_original_metadata(path, modality, expected_count):
    if not path.exists():
        raise FileNotFoundError(f'{modality} sample metadata were not found:\n{path}')
    metadata = pd.read_csv(path, dtype=str, keep_default_na=False)
    if 'Sample ID' not in metadata.columns:
        raise KeyError(f'{path.name} is missing the Sample ID column.')
    metadata['Original Sample ID'] = metadata['Sample ID'].map(canonical_original_sample_id)
    metadata = metadata.drop(columns=['Sample ID'])
    if metadata['Original Sample ID'].duplicated().any():
        duplicates = metadata['Original Sample ID'].value_counts()
        duplicates = duplicates[duplicates > 1]
        display(duplicates.head(30))
        raise ValueError(f'The original {modality} batch metadata contain duplicate Original Sample IDs.')
    if len(metadata) != expected_count:
        raise ValueError(f'The original {modality} batch metadata have {len(metadata)} rows; {expected_count} rows are expected.')
    return metadata
blood_original_metadata = read_original_metadata(BLOOD_METADATA_PATH, 'Blood', 181)
urine_original_metadata = read_original_metadata(URINE_METADATA_PATH, 'Urine', 142)

def create_template_only_if_missing(template_path, original_metadata, modality):
    if template_path.exists():
        print(f'\n[Keep existing file] The {modality} mapping template already exists:')
        
        print('The existing file will be read without overwriting it.')
        return False
    template = original_metadata[['Original Sample ID']].copy()
    template['Current Sample ID'] = ''
    template.to_csv(template_path, index=False, encoding='utf-8-sig')
    
    
    print('Complete Current Sample ID, then rerun this code from the beginning.')
    return True
blood_template_created = create_template_only_if_missing(BLOOD_MAP_PATH, blood_original_metadata, 'Blood')
urine_template_created = create_template_only_if_missing(URINE_MAP_PATH, urine_original_metadata, 'Urine')

def read_filled_mapping(path, modality, expected_sample_ids):
    mapping = pd.read_csv(path, dtype=str, keep_default_na=False)
    required = {'Original Sample ID', 'Current Sample ID'}
    missing_columns = required.difference(mapping.columns)
    if missing_columns:
        raise KeyError(f'{path.name} is missing columns: {sorted(missing_columns)}')
    mapping = mapping[['Original Sample ID', 'Current Sample ID']].copy()
    mapping['Original Sample ID'] = mapping['Original Sample ID'].map(canonical_original_sample_id)
    mapping['Current Sample ID'] = mapping['Current Sample ID'].map(clean_sample_id)
    empty_rows = mapping[(mapping['Original Sample ID'] == '') | (mapping['Current Sample ID'] == '')]
    if not empty_rows.empty:
        print(f'\nThe {modality} mapping template has {len(empty_rows)} incomplete rows.')
        display(empty_rows.head(20))
        return None
    mapping = mapping.drop_duplicates().reset_index(drop=True)
    original_conflicts = mapping.groupby('Original Sample ID')['Current Sample ID'].nunique()
    original_conflicts = original_conflicts[original_conflicts > 1]
    if not original_conflicts.empty:
        conflict_rows = mapping[mapping['Original Sample ID'].isin(original_conflicts.index)]
        display(conflict_rows)
        raise ValueError(f'In {modality}, one Original Sample ID maps to multiple Current Sample IDs.')
    current_conflicts = mapping.groupby('Current Sample ID')['Original Sample ID'].nunique()
    current_conflicts = current_conflicts[current_conflicts > 1]
    if not current_conflicts.empty:
        conflict_rows = mapping[mapping['Current Sample ID'].isin(current_conflicts.index)]
        display(conflict_rows)
        raise ValueError(f'In {modality}, multiple Original Sample IDs map to one Current Sample ID.')
    expected_ids = set((clean_sample_id(value) for value in expected_sample_ids))
    mapped_ids = set(mapping['Current Sample ID'])
    missing_from_mapping = sorted(expected_ids - mapped_ids)
    extra_in_mapping = sorted(mapped_ids - expected_ids)
    if missing_from_mapping or extra_in_mapping:
        mismatch_rows = []
        for sample_id in missing_from_mapping:
            mismatch_rows.append({'Sample ID': sample_id, 'Problem': f'Present in the {modality} matrix but missing from the mapping template'})
        for sample_id in extra_in_mapping:
            mismatch_rows.append({'Sample ID': sample_id, 'Problem': f'Present in the {modality} mapping template but absent from the matrix'})
        mismatch_table = pd.DataFrame(mismatch_rows)
        mismatch_path = OUTPUT_DIR / f"{modality}{get_value('science.tables.result_140', 'result_140.csv')}"
        mismatch_table.to_csv(mismatch_path, index=False, encoding='utf-8-sig')
        print(f'\nThe {modality} mapping and matrix sample names do not fully match.')
        print(f'Present in the matrix but missing from the mapping: {len(missing_from_mapping)}')
        print(f'Present in the mapping but absent from the matrix: {len(extra_in_mapping)}')
        display(mismatch_table.head(30))
        print('Complete discrepancy table:')
        print(mismatch_path)
        raise ValueError(f'Current Sample IDs in the {modality} mapping must exactly match the {modality} matrix sample names.')
    if len(mapping) != len(expected_ids):
        raise ValueError(f'The {modality} mapping has {len(mapping)} rows, whereas the matrix has {len(expected_ids)} samples.')
    return mapping

def build_batch_table(modality, template_path, original_metadata, matrix_sample_ids, expected_cohort_counts):
    mapping = read_filled_mapping(template_path, modality, matrix_sample_ids)
    if mapping is None:
        return None
    merged = mapping.merge(original_metadata, on='Original Sample ID', how='left', validate='one_to_one', indicator=True)
    unmatched = merged[merged['_merge'] != 'both']
    if not unmatched.empty:
        display(unmatched.head(30))
        raise ValueError(f'{modality}: {len(unmatched)} Original Sample IDs do not match the batch metadata.')
    merged = merged.drop(columns=['_merge'])
    merged = merged.rename(columns={'Current Sample ID': 'Sample ID'})
    merged = merged.merge(cohort_mapping[['Sample ID', 'Center', 'Cohort', 'Blood available', 'Urine available']], on='Sample ID', how='left', validate='one_to_one', indicator=True)
    cohort_unmatched = merged[merged['_merge'] != 'both']
    if not cohort_unmatched.empty:
        display(cohort_unmatched.head(30))
        raise ValueError(f'{modality}: {len(cohort_unmatched)} samples do not match cohort information.')
    merged = merged.drop(columns=['_merge'])
    if merged['Sample ID'].duplicated().any():
        raise ValueError(f'The final {modality} batch table contains duplicate Sample IDs.')
    actual_cohort_counts = merged['Cohort'].value_counts(dropna=False).to_dict()
    if actual_cohort_counts != expected_cohort_counts:
        print(f'\nThe final {modality} cohort distribution is unexpected:')
        print('Observed:', actual_cohort_counts)
        print('Expected:', expected_cohort_counts)
        raise ValueError(f'{modality} cohort counts are incorrect after mapping; check Current Sample ID entries for spelling errors.')
    return merged
blood_batch = None
urine_batch = None
if not blood_template_created:
    blood_batch = build_batch_table(modality='Blood', template_path=BLOOD_MAP_PATH, original_metadata=blood_original_metadata, matrix_sample_ids=blood.index, expected_cohort_counts={'Internal': 109, 'Blood_external': 72})
if not urine_template_created:
    urine_batch = build_batch_table(modality='Urine', template_path=URINE_MAP_PATH, original_metadata=urine_original_metadata, matrix_sample_ids=urine.index, expected_cohort_counts={'Internal': 109, 'Urine_external': 33})
if blood_batch is not None:
    blood_output = OUTPUT_DIR / get_value('science.tables.result_141', 'result_141.csv')
    blood_batch.to_csv(blood_output, index=False, encoding='utf-8-sig')
    
    
if urine_batch is not None:
    urine_output = OUTPUT_DIR / get_value('science.tables.result_142', 'result_142.csv')
    urine_batch.to_csv(urine_output, index=False, encoding='utf-8-sig')
    
    
summary_rows = [{'Modality': 'Blood', 'Sample count': blood.shape[0], 'Feature count': blood.shape[1], 'Zero cells': int((blood == 0).sum().sum()), 'Missing cells after zero-to-NA': int(blood_na.isna().sum().sum()), 'Median sample missing rate': float(blood_sample_qc['Missing rate'].median()), 'Maximum sample missing rate': float(blood_sample_qc['Missing rate'].max()), 'Internal features detection >=50%': int(blood_feature_qc['Internal detection >= 50%'].sum()), 'Batch mapping completed': blood_batch is not None}, {'Modality': 'Urine', 'Sample count': urine.shape[0], 'Feature count': urine.shape[1], 'Zero cells': int((urine == 0).sum().sum()), 'Missing cells after zero-to-NA': int(urine_na.isna().sum().sum()), 'Median sample missing rate': float(urine_sample_qc['Missing rate'].median()), 'Maximum sample missing rate': float(urine_sample_qc['Missing rate'].max()), 'Internal features detection >=50%': int(urine_feature_qc['Internal detection >= 50%'].sum()), 'Batch mapping completed': urine_batch is not None}]
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / get_value('science.tables.features_039', 'features_039.csv'), index=False, encoding='utf-8-sig')



print('\nOriginal QC summary:')
display(summary)
print('\nCohort distribution:')
display(cohort_mapping['Cohort'].value_counts())
if blood_batch is not None:
    print('\nBlood batch table:')
    print(blood_batch.shape)
    
    print('\nFinal blood cohort distribution:')
    display(blood_batch['Cohort'].value_counts(dropna=False))
if urine_batch is not None:
    print('\nUrine batch table:')
    print(urine_batch.shape)
    
    print('\nFinal urine cohort distribution:')
    display(urine_batch['Cohort'].value_counts(dropna=False))
if blood_batch is None or urine_batch is None:
    print('\nAt least one mapping template is incomplete.\nExisting templates will not be overwritten.\nComplete Current Sample ID, then rerun this code from the beginning.')




In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
try:
    import openpyxl
    from openpyxl import load_workbook
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.formatting.rule import CellIsRule
    from openpyxl.utils import get_column_letter
except ImportError as exc:
    raise ImportError('openpyxl is missing. Run the following command in the current environment first:\npip install openpyxl') from exc
BASE_DIR = Path(str(project_path(get_value('science.paths.directory_100', 'directory_100'))))
AUDIT_DIR = BASE_DIR / get_value('science.paths.directory_068', 'directory_068')
OUTPUT_XLSX = AUDIT_DIR / get_value('science.tables.metrics_062', 'metrics_062.xlsx')
PATHS = {'QC_summary': AUDIT_DIR / get_value('science.tables.features_039', 'features_039.csv'), 'Cohort_mapping': AUDIT_DIR / get_value('science.tables.result_133', 'result_133.csv'), 'Blood_batch_metadata': AUDIT_DIR / get_value('science.tables.result_141', 'result_141.csv'), 'Urine_batch_metadata': AUDIT_DIR / get_value('science.tables.result_142', 'result_142.csv'), 'Blood_sample_QC': AUDIT_DIR / get_value('science.tables.result_136', 'result_136.csv'), 'Urine_sample_QC': AUDIT_DIR / get_value('science.tables.result_137', 'result_137.csv'), 'Blood_feature_QC': AUDIT_DIR / get_value('science.tables.result_138', 'result_138.csv'), 'Urine_feature_QC': AUDIT_DIR / get_value('science.tables.result_139', 'result_139.csv'), 'Blood_mapping': AUDIT_DIR / get_value('science.tables.result_131', 'result_131.csv'), 'Urine_mapping': AUDIT_DIR / get_value('science.tables.result_132', 'result_132.csv')}

def clean_sample_id(value):
    if pd.isna(value):
        return ''
    text = str(value)
    text = text.replace('\u3000', ' ')
    text = text.strip().strip('"').strip("'").strip()
    if re.fullmatch('\\d+\\.0', text):
        text = text[:-2]
    return text.upper()

def read_required_csv(path, label):
    if not path.exists():
        raise FileNotFoundError(f'The {label} file is missing:\n{path}\nRun the preceding integration-audit code completely first.')
    return pd.read_csv(path, low_memory=False)

def normalize_sample_column(dataframe, column='Sample ID'):
    result = dataframe.copy()
    if column in result.columns:
        result[column] = result[column].map(clean_sample_id)
    return result

def is_blank_series(series):
    return series.isna() | series.astype(str).str.strip().eq('') | series.astype(str).str.strip().str.lower().eq('nan')
tables = {name: read_required_csv(path, name) for name, path in PATHS.items()}
qc_summary = tables['QC_summary'].copy()
cohort_mapping = normalize_sample_column(tables['Cohort_mapping'])
blood_batch = normalize_sample_column(tables['Blood_batch_metadata'])
urine_batch = normalize_sample_column(tables['Urine_batch_metadata'])
blood_sample_qc = normalize_sample_column(tables['Blood_sample_QC'])
urine_sample_qc = normalize_sample_column(tables['Urine_sample_QC'])
blood_feature_qc = tables['Blood_feature_QC'].copy()
urine_feature_qc = tables['Urine_feature_QC'].copy()
blood_mapping = tables['Blood_mapping'].copy()
urine_mapping = tables['Urine_mapping'].copy()
for mapping in [blood_mapping, urine_mapping]:
    if 'Original Sample ID' in mapping.columns:
        mapping['Original Sample ID'] = mapping['Original Sample ID'].map(clean_sample_id)
    if 'Current Sample ID' in mapping.columns:
        mapping['Current Sample ID'] = mapping['Current Sample ID'].map(clean_sample_id)
checks = []

def add_check(category, check_name, actual, expected, passed, detail=''):
    checks.append({'Category': category, 'Check': check_name, 'Status': 'PASS' if passed else 'FAIL', 'Actual': actual, 'Expected': expected, 'Detail': detail})
cohort_counts = cohort_mapping['Cohort'].value_counts(dropna=False).to_dict()
expected_overall = {'Internal': 109, 'Blood_external': 72, 'Urine_external': 33}
for cohort, expected_count in expected_overall.items():
    actual = int(cohort_counts.get(cohort, 0))
    add_check('Overall cohort', f'{cohort} count', actual, expected_count, actual == expected_count)
add_check('Overall cohort', 'Total unique samples', cohort_mapping['Sample ID'].nunique(), 214, cohort_mapping['Sample ID'].nunique() == 214)
add_check('Overall cohort', 'Duplicated Sample ID', int(cohort_mapping['Sample ID'].duplicated().sum()), 0, not cohort_mapping['Sample ID'].duplicated().any())
blood_expected_ids = set(cohort_mapping.loc[pd.to_numeric(cohort_mapping['Blood available'], errors='coerce').fillna(0).astype(int) == 1, 'Sample ID'])
blood_batch_ids = set(blood_batch['Sample ID'])
blood_batch_counts = blood_batch['Cohort'].value_counts(dropna=False).to_dict()
add_check('Blood batch', 'Row count', len(blood_batch), 181, len(blood_batch) == 181)
add_check('Blood batch', 'Unique Sample ID', blood_batch['Sample ID'].nunique(), 181, blood_batch['Sample ID'].nunique() == 181)
add_check('Blood batch', 'Internal count', int(blood_batch_counts.get('Internal', 0)), 109, int(blood_batch_counts.get('Internal', 0)) == 109)
add_check('Blood batch', 'Blood external count', int(blood_batch_counts.get('Blood_external', 0)), 72, int(blood_batch_counts.get('Blood_external', 0)) == 72)
add_check('Blood batch', 'Missing Cohort', int(is_blank_series(blood_batch['Cohort']).sum()), 0, not is_blank_series(blood_batch['Cohort']).any())
add_check('Blood batch', 'Sample set equals Blood matrix set', len(blood_batch_ids.symmetric_difference(blood_expected_ids)), 0, blood_batch_ids == blood_expected_ids, detail='Number shown is the symmetric difference between Batch metadata and Blood matrix sample sets.')
add_check('Blood batch', 'Duplicated Original Sample ID', int(blood_batch['Original Sample ID'].duplicated().sum()), 0, not blood_batch['Original Sample ID'].duplicated().any())
urine_expected_ids = set(cohort_mapping.loc[pd.to_numeric(cohort_mapping['Urine available'], errors='coerce').fillna(0).astype(int) == 1, 'Sample ID'])
urine_batch_ids = set(urine_batch['Sample ID'])
urine_batch_counts = urine_batch['Cohort'].value_counts(dropna=False).to_dict()
add_check('Urine batch', 'Row count', len(urine_batch), 142, len(urine_batch) == 142)
add_check('Urine batch', 'Unique Sample ID', urine_batch['Sample ID'].nunique(), 142, urine_batch['Sample ID'].nunique() == 142)
add_check('Urine batch', 'Internal count', int(urine_batch_counts.get('Internal', 0)), 109, int(urine_batch_counts.get('Internal', 0)) == 109)
add_check('Urine batch', 'Urine external count', int(urine_batch_counts.get('Urine_external', 0)), 33, int(urine_batch_counts.get('Urine_external', 0)) == 33)
add_check('Urine batch', 'Missing Cohort', int(is_blank_series(urine_batch['Cohort']).sum()), 0, not is_blank_series(urine_batch['Cohort']).any())
add_check('Urine batch', 'Sample set equals Urine matrix set', len(urine_batch_ids.symmetric_difference(urine_expected_ids)), 0, urine_batch_ids == urine_expected_ids, detail='Number shown is the symmetric difference between Batch metadata and Urine matrix sample sets.')
add_check('Urine batch', 'Duplicated Original Sample ID', int(urine_batch['Original Sample ID'].duplicated().sum()), 0, not urine_batch['Original Sample ID'].duplicated().any())
batch_columns = ['POS Batch ID', 'NEG Batch ID', 'POS Injection order', 'NEG Injection order']
for modality, dataframe in [('Blood', blood_batch), ('Urine', urine_batch)]:
    for column in batch_columns:
        if column not in dataframe.columns:
            add_check(f'{modality} metadata', f'{column} column exists', 'Missing', 'Present', False)
            continue
        missing_count = int(is_blank_series(dataframe[column]).sum())
        add_check(f'{modality} metadata', f'Missing {column}', missing_count, 0, missing_count == 0)
for modality, mapping, expected_count in [('Blood', blood_mapping, 181), ('Urine', urine_mapping, 142)]:
    add_check(f'{modality} mapping', 'Row count', len(mapping), expected_count, len(mapping) == expected_count)
    for column in ['Original Sample ID', 'Current Sample ID']:
        if column not in mapping.columns:
            add_check(f'{modality} mapping', f'{column} column exists', 'Missing', 'Present', False)
            continue
        blank_count = int(is_blank_series(mapping[column]).sum())
        duplicate_count = int(mapping[column].duplicated().sum())
        add_check(f'{modality} mapping', f'Blank {column}', blank_count, 0, blank_count == 0)
        add_check(f'{modality} mapping', f'Duplicated {column}', duplicate_count, 0, duplicate_count == 0)
add_check('Sample QC', 'Blood sample QC row count', len(blood_sample_qc), 181, len(blood_sample_qc) == 181)
add_check('Sample QC', 'Urine sample QC row count', len(urine_sample_qc), 142, len(urine_sample_qc) == 142)
add_check('Sample QC', 'Blood sample QC unique Sample ID', blood_sample_qc['Sample ID'].nunique(), 181, blood_sample_qc['Sample ID'].nunique() == 181)
add_check('Sample QC', 'Urine sample QC unique Sample ID', urine_sample_qc['Sample ID'].nunique(), 142, urine_sample_qc['Sample ID'].nunique() == 142)
add_check('Feature QC', 'Blood feature count', len(blood_feature_qc), 1081, len(blood_feature_qc) == 1081)
add_check('Feature QC', 'Urine feature count', len(urine_feature_qc), 3041, len(urine_feature_qc) == 3041)
if 'Feature' in blood_feature_qc.columns:
    add_check('Feature QC', 'Blood duplicated feature names', int(blood_feature_qc['Feature'].duplicated().sum()), 0, not blood_feature_qc['Feature'].duplicated().any())
if 'Feature' in urine_feature_qc.columns:
    add_check('Feature QC', 'Urine duplicated feature names', int(urine_feature_qc['Feature'].duplicated().sum()), 0, not urine_feature_qc['Feature'].duplicated().any())
validation_checks = pd.DataFrame(checks)
failed_checks = validation_checks[validation_checks['Status'] == 'FAIL'].copy()
readme_rows = [['Blood / Urine raw matrix audit package', ''], ['Purpose', 'Intermediate audit package before normalization, imputation, log transformation and batch correction.'], ['Internal cohort definition', 'Samples present in both Blood and Urine matrices; sample matching is case-insensitive.'], ['Expected cohort sizes', 'Internal 109; Blood external 72; Urine external 33.'], ['Blood matrix', '181 samples × 1081 nonredundant annotated ion features.'], ['Urine matrix', '142 samples × 3041 nonredundant annotated ion features.'], ['Zero handling', 'Raw zero values were converted to missing values for audit outputs; no imputation was performed.'], ['Not yet performed', 'Sample normalization, urine dilution correction, imputation, log transformation, batch correction, scaling, outcome-linked filtering or modeling.'], ['Raw matrices included?', 'No. The full raw and zero-as-NA matrices remain as local compressed CSV files and are intentionally excluded from this review workbook.'], ['Validation result', 'PASS' if failed_checks.empty else f'FAIL: {len(failed_checks)} checks require review'], ['Output file', str(OUTPUT_XLSX)]]
sheet_order = ['README', 'Validation_checks', 'QC_summary', 'Cohort_mapping', 'Blood_batch_metadata', 'Urine_batch_metadata', 'Blood_sample_QC', 'Urine_sample_QC', 'Blood_feature_QC', 'Urine_feature_QC', 'Blood_mapping', 'Urine_mapping']
with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    pd.DataFrame(readme_rows, columns=['Item', 'Description']).to_excel(writer, sheet_name='README', index=False)
    validation_checks.to_excel(writer, sheet_name='Validation_checks', index=False)
    qc_summary.to_excel(writer, sheet_name='QC_summary', index=False)
    cohort_mapping.to_excel(writer, sheet_name='Cohort_mapping', index=False)
    blood_batch.to_excel(writer, sheet_name='Blood_batch_metadata', index=False)
    urine_batch.to_excel(writer, sheet_name='Urine_batch_metadata', index=False)
    blood_sample_qc.to_excel(writer, sheet_name='Blood_sample_QC', index=False)
    urine_sample_qc.to_excel(writer, sheet_name='Urine_sample_QC', index=False)
    blood_feature_qc.to_excel(writer, sheet_name='Blood_feature_QC', index=False)
    urine_feature_qc.to_excel(writer, sheet_name='Urine_feature_QC', index=False)
    blood_mapping.to_excel(writer, sheet_name='Blood_mapping', index=False)
    urine_mapping.to_excel(writer, sheet_name='Urine_mapping', index=False)
workbook = load_workbook(OUTPUT_XLSX)
workbook.active = workbook.sheetnames.index('README')
header_fill = PatternFill(fill_type='solid', fgColor='1F4E78')
header_font = Font(color='FFFFFF', bold=True)
pass_fill = PatternFill(fill_type='solid', fgColor='C6EFCE')
fail_fill = PatternFill(fill_type='solid', fgColor='FFC7CE')
pass_font = Font(color='006100', bold=True)
fail_font = Font(color='9C0006', bold=True)
thin_gray = Side(style='thin', color='D9E2F3')
body_border = Border(bottom=thin_gray)
percentage_keywords = ('rate', 'percent', '>=')
scientific_keywords = ('intensity',)

def choose_column_width(worksheet, column_index, max_width=38):
    max_length = 0
    for row in worksheet.iter_rows(min_col=column_index, max_col=column_index):
        value = row[0].value
        if value is None:
            continue
        text = str(value)
        max_length = max(max_length, len(text))
    return min(max(max_length + 2, 10), max_width)
for worksheet in workbook.worksheets:
    worksheet.sheet_view.showGridLines = False
    worksheet.freeze_panes = 'A2'
    worksheet.auto_filter.ref = worksheet.dimensions
    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    worksheet.row_dimensions[1].height = 30
    for row in worksheet.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical='top', wrap_text=False)
            cell.border = body_border
    for column_index in range(1, worksheet.max_column + 1):
        width = choose_column_width(worksheet, column_index)
        worksheet.column_dimensions[get_column_letter(column_index)].width = width
    headers = {cell.column: str(cell.value) for cell in worksheet[1] if cell.value is not None}
    for column_index, header in headers.items():
        header_lower = header.lower()
        if any((keyword in header_lower for keyword in percentage_keywords)):
            for cell in worksheet.iter_cols(min_col=column_index, max_col=column_index, min_row=2):
                for target in cell:
                    if isinstance(target.value, (int, float)):
                        target.number_format = '0.00%'
        elif any((keyword in header_lower for keyword in scientific_keywords)):
            for cell in worksheet.iter_cols(min_col=column_index, max_col=column_index, min_row=2):
                for target in cell:
                    if isinstance(target.value, (int, float)):
                        target.number_format = '0.000E+00'
    if worksheet.title == 'README':
        worksheet.column_dimensions['A'].width = 28
        worksheet.column_dimensions['B'].width = 95
        for row in worksheet.iter_rows(min_row=2):
            row[1].alignment = Alignment(vertical='top', wrap_text=True)
    if worksheet.title == 'Validation_checks':
        status_column = None
        for cell in worksheet[1]:
            if cell.value == 'Status':
                status_column = cell.column
                break
        if status_column is not None:
            for row_index in range(2, worksheet.max_row + 1):
                status_cell = worksheet.cell(row=row_index, column=status_column)
                if status_cell.value == 'PASS':
                    status_cell.fill = pass_fill
                    status_cell.font = pass_font
                elif status_cell.value == 'FAIL':
                    status_cell.fill = fail_fill
                    status_cell.font = fail_font
        worksheet.column_dimensions['B'].width = 38
        worksheet.column_dimensions['F'].width = 60
    if worksheet.title in {'Cohort_mapping', 'Blood_batch_metadata', 'Urine_batch_metadata', 'Blood_sample_QC', 'Urine_sample_QC', 'Blood_mapping', 'Urine_mapping'}:
        worksheet.column_dimensions['A'].width = max(worksheet.column_dimensions['A'].width or 10, 22)
workbook.save(OUTPUT_XLSX)






for sheet_name in sheet_order:
    pass
print('\nValidation results:')
print(validation_checks['Status'].value_counts())
if failed_checks.empty:
    print('\nAll key checks are PASS.\nThe Excel file can be uploaded for review.')
else:
    print(f'\nThere are {len(failed_checks)} FAIL items.\nThe Excel file was still generated; review the Validation_checks worksheet first.')
    display(failed_checks)


In [ ]:
from pathlib import Path
import json
import re
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore', category=RuntimeWarning)
BASE_DIR = Path(str(project_path(get_value('science.paths.directory_100', 'directory_100'))))
AUDIT_DIR = BASE_DIR / get_value('science.paths.directory_068', 'directory_068')
OUT_DIR = BASE_DIR / get_value('science.paths.directory_069', 'directory_069')
OUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DICTIONARY = Path(str(project_path(get_value('science.tables.metrics_061', 'metrics_061.xlsx'))))
BLOOD_MATRIX_PATH = AUDIT_DIR / get_value('science.tables.features_037', 'features_037.csv.gz')
URINE_MATRIX_PATH = AUDIT_DIR / get_value('science.tables.features_038', 'features_038.csv.gz')
BLOOD_META_PATH = AUDIT_DIR / get_value('science.tables.result_141', 'result_141.csv')
URINE_META_PATH = AUDIT_DIR / get_value('science.tables.result_142', 'result_142.csv')
DETECTION_THRESHOLD = 0.5
EPS = 1e-08
COMBAT_CONVERGENCE = 0.0001
COMBAT_MAX_ITER = 1000
EXPECTED_INTERNAL_N = 109
EXPECTED_EXTERNAL_N = {'Blood': 72, 'Urine': 33}
EXPECTED_EXTERNAL_LABEL = {'Blood': 'Blood_external', 'Urine': 'Urine_external'}

def clean_sample_id(value):
    if pd.isna(value):
        return ''
    text = str(value).replace('\u3000', ' ').strip().strip('"').strip("'").strip()
    if re.fullmatch('\\d+\\.0', text):
        text = text[:-2]
    return text.upper()

def clean_feature_text(value):
    if pd.isna(value):
        return ''
    text = str(value).replace('\u3000', ' ').strip()
    old = None
    while text != old:
        old = text
        text = text.strip().strip('"').strip("'").strip()
    return text

def make_unique_dot(names):
    counts, used, result = ({}, set(), [])
    for value in names:
        base = clean_feature_text(value) or 'Unnamed_feature'
        n = counts.get(base, 0)
        candidate = base if n == 0 else f'{base}.{n}'
        while candidate in used:
            n += 1
            candidate = f'{base}.{n}'
        counts[base] = n + 1
        used.add(candidate)
        result.append(candidate)
    return result

def read_matrix(path):
    if not path.exists():
        raise FileNotFoundError(f'The matrix file is missing:\n{path}')
    X = pd.read_csv(path, index_col=0, low_memory=False)
    X.index = [clean_sample_id(x) for x in X.index]
    X.columns = [clean_feature_text(x) for x in X.columns]
    X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    if X.index.duplicated().any():
        raise ValueError('The matrix contains duplicate Sample IDs.')
    if pd.Index(X.columns).duplicated().any():
        raise ValueError('The matrix contains duplicate feature column names.')
    return X

def read_metadata(path):
    if not path.exists():
        raise FileNotFoundError(f'Sample metadata are missing:\n{path}')
    meta = pd.read_csv(path, low_memory=False)
    required = ['Sample ID', 'Cohort', 'POS Batch ID', 'NEG Batch ID']
    missing = [x for x in required if x not in meta.columns]
    if missing:
        raise ValueError(f'The metadata are missing required columns: {missing}')
    meta['Sample ID'] = meta['Sample ID'].map(clean_sample_id)
    meta = meta.set_index('Sample ID', drop=False)
    if meta.index.duplicated().any():
        raise ValueError('The metadata contain duplicate Sample IDs.')
    for col in ['POS Batch ID', 'NEG Batch ID']:
        meta[col] = meta[col].astype(str).str.strip().str.replace('\\.0$', '', regex=True)
        if meta[col].isin(['', 'nan', 'None']).any():
            raise ValueError(f'{col} contains missing values.')
    return meta

def load_manifest(modality, matrix_columns):
    sheet = get_value('science.translation.preprocess_001', 'Blood final dictionary') if modality == 'Blood' else get_value('science.translation.preprocess_002', 'Urine final dictionary')
    dic = pd.read_excel(FEATURE_DICTIONARY, sheet_name=sheet)
    required = ['Feature ID', 'Cleaned candidate', 'Ion mode']
    missing = [x for x in required if x not in dic.columns]
    if missing:
        raise ValueError(f'{sheet} is missing required columns: {missing}')
    dic = dic.copy()
    dic['Feature ID'] = dic['Feature ID'].astype(str).str.strip()
    dic['Cleaned candidate'] = dic['Cleaned candidate'].map(clean_feature_text)
    dic['Ion mode'] = dic['Ion mode'].astype(str).str.upper().str.strip()
    if len(dic) != len(matrix_columns):
        raise ValueError(f'{modality}: dictionary rows and matrix features differ in number: {len(dic)} vs {len(matrix_columns)}')
    expected = make_unique_dot(dic['Cleaned candidate'].tolist())
    actual = [clean_feature_text(x) for x in matrix_columns]
    if expected != actual:
        mismatch = pd.DataFrame({'Position': np.arange(1, len(actual) + 1), 'Feature ID': dic['Feature ID'], 'Cleaned candidate': dic['Cleaned candidate'], 'Expected model column name': expected, 'Actual matrix column name': actual, 'Exact match': np.array(expected) == np.array(actual)})
        path = OUT_DIR / f"{modality}{get_value('science.tables.result_143', 'result_143.csv')}"
        mismatch.to_csv(path, index=False, encoding='utf-8-sig')
        raise ValueError(f'{modality}: dictionary and matrix column names do not exactly match; the discrepancy table was saved to: {path}')
    dic.insert(1, 'Model column name', actual)
    dic['Putative annotation'] = dic['Cleaned candidate']
    if 'Display label' in dic.columns:
        dic['Figure display label'] = dic['Display label'].map(clean_feature_text)
    else:
        dic['Figure display label'] = dic['Cleaned candidate']
    duplicate_n = dic.groupby('Cleaned candidate')['Feature ID'].transform('count')
    mask = duplicate_n > 1
    if 'RT (min)' in dic.columns:
        rt = pd.to_numeric(dic['RT (min)'], errors='coerce').round(3).astype(str)
    else:
        rt = pd.Series('NA', index=dic.index)
    dic.loc[mask, 'Figure display label'] = dic.loc[mask, 'Cleaned candidate'] + ' [' + dic.loc[mask, 'Ion mode'] + '; RT=' + rt.loc[mask] + '; ' + dic.loc[mask, 'Feature ID'] + ']'
    if not dic['Ion mode'].isin(['POS', 'NEG']).all():
        raise ValueError(f'{modality} has unrecognized Ion mode values.')
    return dic

def split_cohorts(X, meta, modality):
    if set(X.index) != set(meta.index):
        raise ValueError(f'The {modality} matrix and metadata sample sets differ.')
    internal_ids = meta.index[meta['Cohort'] == 'Internal'].tolist()
    external_label = EXPECTED_EXTERNAL_LABEL[modality]
    external_ids = meta.index[meta['Cohort'] == external_label].tolist()
    if len(internal_ids) != EXPECTED_INTERNAL_N:
        raise ValueError(f'{modality} does not have 109 internal samples.')
    if len(external_ids) != EXPECTED_EXTERNAL_N[modality]:
        raise ValueError(f'{modality} has an incorrect external sample count.')
    return (X.loc[internal_ids].copy(), X.loc[external_ids].copy(), meta.loc[internal_ids].copy(), meta.loc[external_ids].copy())

def zero_to_na(X):
    return X.mask(X <= 0)

def fit_detection_filter(X_internal):
    rate = X_internal.notna().mean(axis=0)
    keep = rate.index[rate >= DETECTION_THRESHOLD].tolist()
    if not keep:
        raise ValueError('No features remain after detection-rate filtering.')
    return (rate, keep)

def fit_half_min(X_internal):
    minimum = X_internal.where(X_internal > 0).min(axis=0, skipna=True)
    bad = minimum[minimum.isna() | ~np.isfinite(minimum) | (minimum <= 0)]
    if len(bad):
        raise ValueError(f'The half-minimum value cannot be calculated: {bad.index.tolist()[:20]}')
    return minimum * 0.5

def apply_imputer(X, values):
    out = X.loc[:, values.index].fillna(values)
    if out.isna().any().any() or (out <= 0).any().any():
        raise ValueError('NA or nonpositive values remain after imputation.')
    return out

def fit_pqn_reference(X_internal):
    ref = X_internal.median(axis=0)
    bad = ref[ref.isna() | ~np.isfinite(ref) | (ref <= 0)]
    if len(bad):
        raise ValueError(f'The PQN reference spectrum has invalid values: {bad.index.tolist()[:20]}')
    return ref

def apply_pqn(X, reference):
    ratios = X.loc[:, reference.index].div(reference, axis=1)
    factor = ratios.median(axis=1)
    bad = factor[factor.isna() | ~np.isfinite(factor) | (factor <= 0)]
    if len(bad):
        raise ValueError(f'Samples with invalid PQN factors: {bad.index.tolist()[:20]}')
    out = X.loc[:, reference.index].div(factor, axis=0)
    return (out, factor)

def _aprior(delta, eps=EPS):
    m = max(float(np.mean(delta)), eps)
    s2 = max(float(np.var(delta, ddof=1)), eps)
    return (2.0 * s2 + m * m) / s2

def _bprior(delta, eps=EPS):
    m = max(float(np.mean(delta)), eps)
    s2 = max(float(np.var(delta, ddof=1)), eps)
    return (m * s2 + m ** 3) / s2

def _posterior(s_batch, g_hat, d_hat, g_bar, tau2):
    n = s_batch.shape[1]
    g_old = g_hat.copy()
    d_old = np.maximum(d_hat.copy(), EPS)
    tau2 = max(float(tau2), EPS)
    a, b = (_aprior(d_hat), _bprior(d_hat))
    for _ in range(COMBAT_MAX_ITER):
        g_new = (tau2 * n * g_hat + d_old * g_bar) / (tau2 * n + d_old)
        sum_sq = np.sum((s_batch - g_new[:, None]) ** 2, axis=1)
        d_new = (0.5 * sum_sq + b) / (n / 2.0 + a - 1.0)
        d_new = np.maximum(d_new, EPS)
        change_g = np.max(np.abs(g_new - g_old) / np.maximum(np.abs(g_old), EPS))
        change_d = np.max(np.abs(d_new - d_old) / np.maximum(np.abs(d_old), EPS))
        g_old, d_old = (g_new, d_new)
        if max(float(change_g), float(change_d)) < COMBAT_CONVERGENCE:
            break
    return (g_old, d_old)

def fit_combat(X_internal, batch_series):
    batch = batch_series.reindex(X_internal.index).astype(str)
    levels = list(pd.unique(batch))
    if len(levels) < 2:
        raise ValueError('The internal cohort has only one batch; ComBat cannot be applied.')
    indices = {level: np.where(batch.to_numpy() == level)[0] for level in levels}
    for level, idx in indices.items():
        if len(idx) < 2:
            raise ValueError(f'Batch {level} has fewer than two internal samples.')
    data = X_internal.to_numpy(float).T
    n_samples = data.shape[1]
    design = np.column_stack([(batch.to_numpy() == level).astype(float) for level in levels])
    beta_hat = np.linalg.pinv(design) @ data.T
    sizes = np.array([len(indices[level]) for level in levels], float)
    grand_mean = ((sizes / n_samples)[:, None] * beta_hat).sum(axis=0)
    fitted_batch_mean = (design @ beta_hat).T
    var_pooled = np.mean((data - fitted_batch_mean) ** 2, axis=1)
    var_pooled = np.maximum(var_pooled, EPS)
    sqrt_var = np.sqrt(var_pooled)
    standardized = (data - grand_mean[:, None]) / sqrt_var[:, None]
    gamma_hat, delta_hat = ([], [])
    for level in levels:
        block = standardized[:, indices[level]]
        gamma_hat.append(block.mean(axis=1))
        delta_hat.append(np.maximum(block.var(axis=1, ddof=1), EPS))
    gamma_hat = np.vstack(gamma_hat)
    delta_hat = np.vstack(delta_hat)
    gamma_bar = gamma_hat.mean(axis=1)
    tau2 = np.maximum(gamma_hat.var(axis=1, ddof=1), EPS)
    gamma_star = np.zeros_like(gamma_hat)
    delta_star = np.zeros_like(delta_hat)
    for i, level in enumerate(levels):
        gamma_star[i], delta_star[i] = _posterior(standardized[:, indices[level]], gamma_hat[i], delta_hat[i], gamma_bar[i], tau2[i])
    params = {'feature_names': list(X_internal.columns), 'batch_levels': levels, 'grand_mean': grand_mean, 'var_pooled': var_pooled, 'gamma_star': gamma_star, 'delta_star': delta_star}
    return (apply_combat(X_internal, batch, params), params)

def apply_combat(X, batch_series, params):
    X = X.loc[:, params['feature_names']]
    batch = batch_series.reindex(X.index).astype(str)
    unseen = sorted(set(batch.unique()) - set(params['batch_levels']))
    if unseen:
        raise ValueError(f'The external cohort has batches not observed internally; frozen parameters cannot be applied: {unseen}')
    data = X.to_numpy(float).T
    grand = np.asarray(params['grand_mean'], float)
    var = np.asarray(params['var_pooled'], float)
    gamma = np.asarray(params['gamma_star'], float)
    delta = np.asarray(params['delta_star'], float)
    sqrt_var = np.sqrt(np.maximum(var, EPS))
    standardized = (data - grand[:, None]) / sqrt_var[:, None]
    adjusted = standardized.copy()
    for i, level in enumerate(params['batch_levels']):
        idx = np.where(batch.to_numpy() == level)[0]
        if len(idx):
            adjusted[:, idx] = (standardized[:, idx] - gamma[i, :, None]) / np.sqrt(np.maximum(delta[i, :, None], EPS))
    adjusted = adjusted * sqrt_var[:, None] + grand[:, None]
    return pd.DataFrame(adjusted.T, index=X.index, columns=X.columns)

def save_combat_params(params, path):
    np.savez_compressed(path, feature_names=np.asarray(params['feature_names'], dtype=str), batch_levels=np.asarray(params['batch_levels'], dtype=str), grand_mean=np.asarray(params['grand_mean'], float), var_pooled=np.asarray(params['var_pooled'], float), gamma_star=np.asarray(params['gamma_star'], float), delta_star=np.asarray(params['delta_star'], float))

def process_mode(modality, mode, X_int_na, X_ext_na, meta_int, meta_ext, manifest):
    features = manifest.loc[manifest['Ion mode'] == mode, 'Model column name'].tolist()
    X_int = X_int_na.loc[:, features]
    X_ext = X_ext_na.loc[:, features]
    detection_rate, keep = fit_detection_filter(X_int)
    X_int, X_ext = (X_int.loc[:, keep], X_ext.loc[:, keep])
    impute_values = fit_half_min(X_int)
    X_int_imp = apply_imputer(X_int, impute_values)
    X_ext_imp = apply_imputer(X_ext, impute_values)
    pqn_reference = fit_pqn_reference(X_int_imp)
    X_int_pqn, factor_int = apply_pqn(X_int_imp, pqn_reference)
    X_ext_pqn, factor_ext = apply_pqn(X_ext_imp, pqn_reference)
    X_int_log2 = np.log2(X_int_pqn)
    X_ext_log2 = np.log2(X_ext_pqn)
    batch_col = f'{mode} Batch ID'
    X_int_combat, combat_params = fit_combat(X_int_log2, meta_int[batch_col])
    X_ext_combat = apply_combat(X_ext_log2, meta_ext[batch_col], combat_params)
    save_combat_params(combat_params, OUT_DIR / f"{modality}_{mode}{get_value('science.tables.result_144', 'result_144.npz')}")
    sd_post = X_int_combat.std(axis=0, ddof=0)
    final_features = sd_post.index[np.isfinite(sd_post) & (sd_post > EPS)].tolist()
    removed_zero_var = sorted(set(X_int_combat.columns) - set(final_features))
    X_int_combat = X_int_combat.loc[:, final_features]
    X_ext_combat = X_ext_combat.loc[:, final_features]
    z_mean = X_int_combat.mean(axis=0)
    z_sd = X_int_combat.std(axis=0, ddof=0)
    X_int_z = X_int_combat.sub(z_mean, axis=1).div(z_sd, axis=1)
    X_ext_z = X_ext_combat.sub(z_mean, axis=1).div(z_sd, axis=1)
    if X_int_z.isna().any().any() or X_ext_z.isna().any().any():
        raise ValueError(f'The final {modality}-{mode} matrix still contains NA values.')
    parameters = pd.DataFrame({'Modality': modality, 'Ion mode': mode, 'Model column name': final_features, 'Internal detection rate': detection_rate.loc[final_features].values, 'Half-min imputation value': impute_values.loc[final_features].values, 'PQN internal reference': pqn_reference.loc[final_features].values, 'Internal post-ComBat mean': z_mean.loc[final_features].values, 'Internal post-ComBat SD': z_sd.loc[final_features].values})
    detection_audit = pd.DataFrame({'Modality': modality, 'Ion mode': mode, 'Model column name': detection_rate.index, 'Internal detection rate': detection_rate.values, 'Pass detection filter': detection_rate.values >= DETECTION_THRESHOLD, 'Removed after ComBat zero-variance filter': [x in removed_zero_var for x in detection_rate.index]})
    pqn_factors = pd.concat([pd.DataFrame({'Sample ID': factor_int.index, 'Modality': modality, 'Ion mode': mode, 'Cohort': 'Internal', 'PQN dilution factor': factor_int.values}), pd.DataFrame({'Sample ID': factor_ext.index, 'Modality': modality, 'Ion mode': mode, 'Cohort': EXPECTED_EXTERNAL_LABEL[modality], 'PQN dilution factor': factor_ext.values})], ignore_index=True)
    return {'internal_log2': X_int_log2, 'external_log2': X_ext_log2, 'internal_combat': X_int_combat, 'external_combat': X_ext_combat, 'internal_z': X_int_z, 'external_z': X_ext_z, 'parameters': parameters, 'detection_audit': detection_audit, 'pqn_factors': pqn_factors}

def process_modality(modality, matrix_path, metadata_path):
    
    
    
    X = read_matrix(matrix_path)
    meta = read_metadata(metadata_path)
    manifest = load_manifest(modality, X.columns)
    X_int, X_ext, meta_int, meta_ext = split_cohorts(X, meta, modality)
    X_int_na = zero_to_na(X_int)
    X_ext_na = zero_to_na(X_ext)
    result = {mode: process_mode(modality, mode, X_int_na, X_ext_na, meta_int, meta_ext, manifest) for mode in ['POS', 'NEG']}
    original_order = manifest['Model column name'].tolist()
    final_set = set(result['POS']['internal_z'].columns) | set(result['NEG']['internal_z'].columns)
    final_order = [x for x in original_order if x in final_set]

    def combine(stage):
        out = pd.concat([result['POS'][stage], result['NEG'][stage]], axis=1)
        return out.loc[:, [x for x in final_order if x in out.columns]]
    X_int_log2 = combine('internal_log2')
    X_ext_log2 = combine('external_log2')
    X_int_combat = combine('internal_combat')
    X_ext_combat = combine('external_combat')
    X_int_z = combine('internal_z')
    X_ext_z = combine('external_z')
    X_all_z = pd.concat([X_int_z, X_ext_z], axis=0)
    if len(X_int_z) != EXPECTED_INTERNAL_N or len(X_ext_z) != EXPECTED_EXTERNAL_N[modality]:
        raise ValueError(f'{modality} has an incorrect final sample count.')
    if list(X_int_z.columns) != list(X_ext_z.columns):
        raise ValueError(f'The final {modality} feature columns differ between the internal and external matrices.')
    X_int_z.to_csv(OUT_DIR / f"{modality}{get_value('science.tables.features_025', 'features_025.csv.gz')}", compression='gzip', encoding='utf-8-sig')
    X_ext_z.to_csv(OUT_DIR / f"{modality}{get_value('science.tables.features_040', 'features_040.csv.gz')}", compression='gzip', encoding='utf-8-sig')
    X_all_z.to_csv(OUT_DIR / f"{modality}{get_value('science.tables.features_041', 'features_041.csv.gz')}", compression='gzip', encoding='utf-8-sig')
    X_int_combat.to_csv(OUT_DIR / f"{modality}{get_value('science.tables.result_069', 'result_069.csv.gz')}", compression='gzip', encoding='utf-8-sig')
    X_ext_combat.to_csv(OUT_DIR / f"{modality}{get_value('science.tables.result_145', 'result_145.csv.gz')}", compression='gzip', encoding='utf-8-sig')
    X_int_log2.to_csv(OUT_DIR / f"{modality}{get_value('science.tables.result_068', 'result_068.csv.gz')}", compression='gzip', encoding='utf-8-sig')
    final_manifest = manifest.loc[manifest['Model column name'].isin(final_order)].copy()
    final_manifest['Final feature order'] = final_manifest['Model column name'].map({x: i + 1 for i, x in enumerate(final_order)})
    final_manifest = final_manifest.sort_values('Final feature order')
    final_manifest.to_csv(OUT_DIR / f"{modality}{get_value('science.tables.result_071', 'result_071.csv')}", index=False, encoding='utf-8-sig')
    parameters = pd.concat([result['POS']['parameters'], result['NEG']['parameters']], ignore_index=True)
    parameters.to_csv(OUT_DIR / f"{modality}{get_value('science.tables.result_146', 'result_146.csv.gz')}", index=False, compression='gzip', encoding='utf-8-sig')
    detection = pd.concat([result['POS']['detection_audit'], result['NEG']['detection_audit']], ignore_index=True)
    detection.to_csv(OUT_DIR / f"{modality}{get_value('science.tables.result_147', 'result_147.csv')}", index=False, encoding='utf-8-sig')
    pqn = pd.concat([result['POS']['pqn_factors'], result['NEG']['pqn_factors']], ignore_index=True)
    pqn.to_csv(OUT_DIR / f"{modality}{get_value('science.tables.result_148', 'result_148.csv')}", index=False, encoding='utf-8-sig')
    sample_qc = pd.DataFrame({'Sample ID': X.index, 'Modality': modality, 'Cohort': meta.loc[X.index, 'Cohort'].values, 'Raw zero/NA rate': zero_to_na(X).isna().mean(axis=1).values, 'Raw log10 total signal': np.log10(X.clip(lower=0).sum(axis=1).replace(0, np.nan)).values})
    sample_qc.to_csv(OUT_DIR / f"{modality}{get_value('science.tables.result_072', 'result_072.csv')}", index=False, encoding='utf-8-sig')
    summary = {'Modality': modality, 'Raw samples': X.shape[0], 'Internal samples': X_int_z.shape[0], 'External samples': X_ext_z.shape[0], 'Raw features': X.shape[1], 'Features after detection filter': int(detection['Pass detection filter'].sum()), 'Features removed after ComBat as zero variance': int(detection['Removed after ComBat zero-variance filter'].sum()), 'Final model-ready features': X_int_z.shape[1], 'Internal missing cells': int(X_int_z.isna().sum().sum()), 'External missing cells': int(X_ext_z.isna().sum().sum()), 'Internal mean max abs': float(np.abs(X_int_z.mean(axis=0)).max()), 'Internal SD min': float(X_int_z.std(axis=0, ddof=0).min()), 'Internal SD max': float(X_int_z.std(axis=0, ddof=0).max())}
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return {'summary': summary, 'pqn': pqn, 'sample_qc': sample_qc}
blood = process_modality('Blood', BLOOD_MATRIX_PATH, BLOOD_META_PATH)
urine = process_modality('Urine', URINE_MATRIX_PATH, URINE_META_PATH)
pd.DataFrame([blood['summary'], urine['summary']]).to_csv(OUT_DIR / get_value('science.tables.metrics_041', 'metrics_041.csv'), index=False, encoding='utf-8-sig')
pd.concat([blood['pqn'], urine['pqn']], ignore_index=True).to_csv(OUT_DIR / get_value('science.tables.result_149', 'result_149.csv'), index=False, encoding='utf-8-sig')
pd.concat([blood['sample_qc'], urine['sample_qc']], ignore_index=True).to_csv(OUT_DIR / get_value('science.tables.result_150', 'result_150.csv'), index=False, encoding='utf-8-sig')





print(get_value('science.tables.features_042', 'features_042.csv.gz'))
print(get_value('science.tables.features_043', 'features_043.csv.gz'))
print(get_value('science.tables.features_044', 'features_044.csv.gz'))
print(get_value('science.tables.features_045', 'features_045.csv.gz'))

print(get_value('science.tables.result_151', 'result_151.csv.gz'))
print(get_value('science.tables.result_152', 'result_152.csv.gz'))



In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr, chi2
BASE_DIR = Path(str(project_path(get_value('science.paths.directory_100', 'directory_100'))))
DATA_DIR = BASE_DIR / get_value('science.paths.directory_069', 'directory_069')
META_DIR = BASE_DIR / get_value('science.paths.directory_068', 'directory_068')
OUT_DIR = DATA_DIR / get_value('science.paths.directory_101', 'directory_101')
OUT_DIR.mkdir(parents=True, exist_ok=True)
FILES = {'Blood': {'pre': DATA_DIR / get_value('science.tables.result_153', 'result_153.csv.gz'), 'post': DATA_DIR / get_value('science.tables.result_154', 'result_154.csv.gz'), 'metadata': META_DIR / get_value('science.tables.result_141', 'result_141.csv'), 'manifest': DATA_DIR / get_value('science.tables.result_155', 'result_155.csv')}, 'Urine': {'pre': DATA_DIR / get_value('science.tables.result_156', 'result_156.csv.gz'), 'post': DATA_DIR / get_value('science.tables.result_157', 'result_157.csv.gz'), 'metadata': META_DIR / get_value('science.tables.result_142', 'result_142.csv'), 'manifest': DATA_DIR / get_value('science.tables.result_158', 'result_158.csv')}}
RANDOM_STATE = 20260727
MAX_PCA_COMPONENTS = 20
MAX_OUTLIER_PCS = 5
OUTLIER_P = 0.999
MAX_OUTLIER_LABELS = 10
FIG_DPI = 300

def clean_id(value):
    return str(value).strip()

def read_matrix(path):
    if not path.exists():
        raise FileNotFoundError(f'The matrix file is missing:\n{path}')
    X = pd.read_csv(path, index_col=0, low_memory=False)
    X.index = X.index.map(clean_id)
    X.columns = X.columns.astype(str).str.strip()
    if X.index.duplicated().any():
        duplicated = X.index[X.index.duplicated()].tolist()
        raise ValueError(f'{path.name} contains duplicate Sample IDs, for example: {duplicated[:10]}')
    if X.columns.duplicated().any():
        duplicated = X.columns[X.columns.duplicated()].tolist()
        raise ValueError(f'{path.name} contains duplicate feature names, for example: {duplicated[:10]}')
    X = X.apply(pd.to_numeric, errors='coerce')
    values = X.to_numpy(dtype=float)
    if not np.isfinite(values).all():
        bad_count = int((~np.isfinite(values)).sum())
        raise ValueError(f'{path.name} contains {bad_count} NA or Inf values; check upstream preprocessing first.')
    return X

def read_metadata(path):
    if not path.exists():
        raise FileNotFoundError(f'Batch metadata are missing:\n{path}')
    meta = pd.read_csv(path, low_memory=False)
    required = ['Sample ID', 'POS Batch ID', 'NEG Batch ID']
    missing = [column for column in required if column not in meta.columns]
    if missing:
        raise ValueError(f'{path.name} is missing required columns: {missing}')
    meta['Sample ID'] = meta['Sample ID'].map(clean_id)
    for column in ['POS Batch ID', 'NEG Batch ID']:
        meta[column] = meta[column].astype(str).str.strip().str.replace('\\.0$', '', regex=True)
        invalid = meta[column].isin(['', 'nan', 'None'])
        if invalid.any():
            bad_ids = meta.loc[invalid, 'Sample ID'].tolist()
            raise ValueError(f'{path.name}: {column} contains missing values for these samples: {bad_ids[:10]}')
    meta = meta.set_index('Sample ID', drop=False)
    if meta.index.duplicated().any():
        duplicated = meta.index[meta.index.duplicated()].tolist()
        raise ValueError(f'{path.name} contains duplicate Sample IDs, for example: {duplicated[:10]}')
    return meta

def read_manifest(path):
    if not path.exists():
        raise FileNotFoundError(f'The feature manifest is missing:\n{path}')
    manifest = pd.read_csv(path, low_memory=False)
    required = ['Model column name', 'Ion mode']
    missing = [column for column in required if column not in manifest.columns]
    if missing:
        raise ValueError(f'{path.name} is missing required columns: {missing}')
    manifest = manifest.copy()
    manifest['Model column name'] = manifest['Model column name'].astype(str).str.strip()
    manifest['Ion mode'] = manifest['Ion mode'].astype(str).str.upper().str.strip()
    invalid_modes = sorted(set(manifest['Ion mode']) - {'POS', 'NEG'})
    if invalid_modes:
        raise ValueError(f'{path.name} has unrecognized Ion mode values: {invalid_modes}')
    return manifest

def standardize_for_pca(X):
    feature_sd = X.std(axis=0, ddof=0)
    keep = np.isfinite(feature_sd) & (feature_sd > 1e-12)
    if keep.sum() < 2:
        raise ValueError('Fewer than two valid features have nonzero variance; PCA cannot be performed.')
    X_used = X.loc[:, keep]
    scaler = StandardScaler(with_mean=True, with_std=True)
    Z = scaler.fit_transform(X_used)
    return (Z, X_used.columns.tolist())

def run_pca(Z):
    n_components = min(MAX_PCA_COMPONENTS, Z.shape[0] - 1, Z.shape[1])
    if n_components < 2:
        raise ValueError('PCA has fewer than two valid components.')
    pca = PCA(n_components=n_components, svd_solver='full')
    scores = pca.fit_transform(Z)
    return (pca, scores)

def calculate_batch_r2(Z, batch):
    batch = np.asarray(batch).astype(str)
    grand_center = Z.mean(axis=0)
    total_ss = np.sum((Z - grand_center) ** 2)
    between_ss = 0.0
    for level in pd.unique(batch):
        index = np.where(batch == level)[0]
        group_center = Z[index].mean(axis=0)
        between_ss += len(index) * np.sum((group_center - grand_center) ** 2)
    if total_ss <= 0:
        return np.nan
    return float(between_ss / total_ss)

def detect_batchwise_outliers(scores, sample_ids, batch, modality, ion_mode, stage):
    sample_ids = np.asarray(sample_ids).astype(str)
    batch = np.asarray(batch).astype(str)
    records = []
    for level in pd.unique(batch):
        group_index = np.where(batch == level)[0]
        n_group = len(group_index)
        if n_group < 6:
            for idx in group_index:
                records.append({'Sample ID': sample_ids[idx], 'Modality': modality, 'Ion mode': ion_mode, 'Stage': stage, 'Batch': level, 'Batch sample count': n_group, 'PCs used': np.nan, 'Robust distance squared': np.nan, 'Threshold': np.nan, 'Outlier': False, 'Method': 'Batch size < 6; not tested'})
            continue
        n_pcs = min(MAX_OUTLIER_PCS, scores.shape[1], max(2, (n_group - 1) // 3))
        group_scores = scores[group_index, :n_pcs]
        method = 'Minimum covariance determinant'
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                robust_cov = MinCovDet(random_state=RANDOM_STATE).fit(group_scores)
            distance_squared = robust_cov.mahalanobis(group_scores)
        except Exception:
            method = 'Classical covariance fallback'
            center = np.mean(group_scores, axis=0)
            covariance = np.cov(group_scores, rowvar=False)
            covariance_inverse = np.linalg.pinv(np.atleast_2d(covariance))
            centered = group_scores - center
            distance_squared = np.einsum('ij,jk,ik->i', centered, covariance_inverse, centered)
        threshold = float(chi2.ppf(OUTLIER_P, df=n_pcs))
        outlier_flags = distance_squared > threshold
        for local_i, global_i in enumerate(group_index):
            records.append({'Sample ID': sample_ids[global_i], 'Modality': modality, 'Ion mode': ion_mode, 'Stage': stage, 'Batch': level, 'Batch sample count': n_group, 'PCs used': n_pcs, 'Robust distance squared': float(distance_squared[local_i]), 'Threshold': threshold, 'Outlier': bool(outlier_flags[local_i]), 'Method': method})
    result = pd.DataFrame(records)
    return result

def within_batch_distance_preservation(Z_pre, Z_post, batch):
    batch = np.asarray(batch).astype(str)
    distance_pre = []
    distance_post = []
    for level in pd.unique(batch):
        index = np.where(batch == level)[0]
        if len(index) < 3:
            continue
        distance_pre.extend(pdist(Z_pre[index], metric='euclidean'))
        distance_post.extend(pdist(Z_post[index], metric='euclidean'))
    if len(distance_pre) < 3:
        return (np.nan, np.nan)
    rho, p_value = spearmanr(distance_pre, distance_post)
    return (float(rho), float(p_value))

def feature_sd_ratio(X_pre, X_post):
    sd_pre = X_pre.std(axis=0, ddof=0)
    sd_post = X_post.std(axis=0, ddof=0)
    valid = np.isfinite(sd_pre) & np.isfinite(sd_post) & (sd_pre > 1e-12)
    ratios = sd_post.loc[valid] / sd_pre.loc[valid]
    return {'median': float(ratios.median()), 'p10': float(ratios.quantile(0.1)), 'p90': float(ratios.quantile(0.9)), 'minimum': float(ratios.min()), 'maximum': float(ratios.max())}

def make_qc_flag(r2_pre, r2_post, distance_rho, sd_ratio_median, outliers_pre, outliers_post):
    warnings_list = []
    if np.isfinite(r2_pre) and np.isfinite(r2_post) and (r2_post > r2_pre + 0.005):
        warnings_list.append('Batch separation increased')
    if np.isfinite(r2_pre) and np.isfinite(r2_post) and (r2_pre >= 0.02) and (r2_post > 0.8 * r2_pre):
        warnings_list.append('Batch separation only weakly reduced')
    if np.isfinite(distance_rho) and distance_rho < 0.7:
        warnings_list.append('Possible structure distortion')
    if np.isfinite(sd_ratio_median) and sd_ratio_median < 0.5:
        warnings_list.append('Possible variance over-shrinkage')
    if outliers_post > max(outliers_pre + 2, 3):
        warnings_list.append('More outliers after ComBat')
    if not warnings_list:
        return 'No obvious technical warning'
    return '; '.join(warnings_list)

def plot_pca_panel(ax, scores, pca, batch, outlier_table, color_map, title, batch_r2):
    batch = pd.Series(np.asarray(batch).astype(str))
    for level in color_map:
        mask = batch.to_numpy() == level
        ax.scatter(scores[mask, 0], scores[mask, 1], s=46, alpha=0.82, color=color_map[level], label=f'Batch {level} (n={int(mask.sum())})', linewidths=0.3, edgecolors='white')
    flagged = outlier_table.loc[outlier_table['Outlier']].copy()
    if not flagged.empty:
        sample_to_position = {sample_id: position for position, sample_id in enumerate(outlier_table['Sample ID'])}
        flagged['Position'] = flagged['Sample ID'].map(sample_to_position)
        flagged = flagged.sort_values('Robust distance squared', ascending=False)
        positions = flagged['Position'].dropna().astype(int).to_numpy()
        ax.scatter(scores[positions, 0], scores[positions, 1], s=105, facecolors='none', edgecolors='black', linewidths=1.4, marker='o', label='Flagged outlier')
        for _, row in flagged.head(MAX_OUTLIER_LABELS).iterrows():
            position = int(row['Position'])
            ax.annotate(row['Sample ID'], (scores[position, 0], scores[position, 1]), xytext=(4, 4), textcoords='offset points', fontsize=7)
    pc1 = pca.explained_variance_ratio_[0] * 100
    pc2 = pca.explained_variance_ratio_[1] * 100
    ax.set_title(f'{title}\nBatch R² = {batch_r2:.4f}', fontsize=11)
    ax.set_xlabel(f'PC1 ({pc1:.1f}%)')
    ax.set_ylabel(f'PC2 ({pc2:.1f}%)')
    ax.axhline(0, linewidth=0.6, alpha=0.3)
    ax.axvline(0, linewidth=0.6, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc='best')

def analyze_modality(modality, config):
    
    
    
    X_pre_all = read_matrix(config['pre'])
    X_post_all = read_matrix(config['post'])
    metadata = read_metadata(config['metadata'])
    manifest = read_manifest(config['manifest'])
    pre_samples = set(X_pre_all.index)
    post_samples = set(X_post_all.index)
    if pre_samples != post_samples:
        only_pre = sorted(pre_samples - post_samples)
        only_post = sorted(post_samples - pre_samples)
        raise ValueError(f'{modality} sample sets differ before and after correction.\nPresent only before correction: {only_pre[:10]}\nPresent only after correction: {only_post[:10]}')
    missing_metadata = sorted(pre_samples - set(metadata.index))
    if missing_metadata:
        raise ValueError(f'{modality} has samples that do not match batch metadata: {missing_metadata[:10]}')
    sample_order = X_pre_all.index.tolist()
    X_post_all = X_post_all.loc[sample_order]
    metadata = metadata.loc[sample_order]
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12.5, 10.0))
    summary_records = []
    score_tables = []
    outlier_tables = []
    for row_index, ion_mode in enumerate(['POS', 'NEG']):
        batch_column = f'{ion_mode} Batch ID'
        batch = metadata[batch_column].astype(str).to_numpy()
        manifest_features = manifest.loc[manifest['Ion mode'] == ion_mode, 'Model column name'].tolist()
        common_features = [feature for feature in manifest_features if feature in X_pre_all.columns and feature in X_post_all.columns]
        if len(common_features) < 2:
            raise ValueError(f'{modality}-{ion_mode} has fewer than two shared valid features.')
        X_pre = X_pre_all.loc[sample_order, common_features]
        X_post = X_post_all.loc[sample_order, common_features]
        Z_pre, used_pre_features = standardize_for_pca(X_pre)
        Z_post, used_post_features = standardize_for_pca(X_post)
        if used_pre_features != used_post_features:
            common_after_sd_check = [feature for feature in used_pre_features if feature in set(used_post_features)]
            X_pre = X_pre.loc[:, common_after_sd_check]
            X_post = X_post.loc[:, common_after_sd_check]
            Z_pre, used_pre_features = standardize_for_pca(X_pre)
            Z_post, used_post_features = standardize_for_pca(X_post)
        pca_pre, scores_pre = run_pca(Z_pre)
        pca_post, scores_post = run_pca(Z_post)
        r2_pre = calculate_batch_r2(Z_pre, batch)
        r2_post = calculate_batch_r2(Z_post, batch)
        outlier_pre = detect_batchwise_outliers(scores=scores_pre, sample_ids=sample_order, batch=batch, modality=modality, ion_mode=ion_mode, stage='Before ComBat')
        outlier_post = detect_batchwise_outliers(scores=scores_post, sample_ids=sample_order, batch=batch, modality=modality, ion_mode=ion_mode, stage='After ComBat')
        outlier_tables.extend([outlier_pre, outlier_post])
        n_outlier_pre = int(outlier_pre['Outlier'].sum())
        n_outlier_post = int(outlier_post['Outlier'].sum())
        distance_rho, distance_p = within_batch_distance_preservation(Z_pre, Z_post, batch)
        sd_ratios = feature_sd_ratio(X_pre, X_post)
        qc_flag = make_qc_flag(r2_pre=r2_pre, r2_post=r2_post, distance_rho=distance_rho, sd_ratio_median=sd_ratios['median'], outliers_pre=n_outlier_pre, outliers_post=n_outlier_post)
        batch_counts = pd.Series(batch).value_counts().sort_index()
        batch_count_text = '; '.join((f'{level}:{count}' for level, count in batch_counts.items()))
        summary_records.append({'Modality': modality, 'Ion mode': ion_mode, 'N samples': len(sample_order), 'N features': len(used_pre_features), 'N batches': int(batch_counts.size), 'Batch counts': batch_count_text, 'Batch R2 before': r2_pre, 'Batch R2 after': r2_post, 'Batch R2 absolute change': r2_post - r2_pre, 'Batch R2 retained proportion': r2_post / r2_pre if r2_pre > 0 else np.nan, 'Within-batch distance Spearman rho': distance_rho, 'Within-batch distance p': distance_p, 'Median feature SD ratio post/pre': sd_ratios['median'], 'Feature SD ratio P10': sd_ratios['p10'], 'Feature SD ratio P90': sd_ratios['p90'], 'Outliers before': n_outlier_pre, 'Outliers after': n_outlier_post, 'QC flag (heuristic)': qc_flag})
        for stage, scores, pca_object, outlier_table in [('Before ComBat', scores_pre, pca_pre, outlier_pre), ('After ComBat', scores_post, pca_post, outlier_post)]:
            n_saved_pcs = min(5, scores.shape[1])
            score_table = pd.DataFrame(scores[:, :n_saved_pcs], index=sample_order, columns=[f'PC{i + 1}' for i in range(n_saved_pcs)])
            score_table.insert(0, 'Sample ID', score_table.index)
            score_table.insert(1, 'Modality', modality)
            score_table.insert(2, 'Ion mode', ion_mode)
            score_table.insert(3, 'Stage', stage)
            score_table.insert(4, 'Batch', batch)
            outlier_map = outlier_table.set_index('Sample ID')['Outlier']
            score_table['Outlier'] = score_table['Sample ID'].map(outlier_map).fillna(False).astype(bool)
            score_tables.append(score_table.reset_index(drop=True))
        batch_levels = sorted(pd.unique(batch), key=str)
        color_map = {level: plt.get_cmap('tab10')(index % 10) for index, level in enumerate(batch_levels)}
        plot_pca_panel(ax=axes[row_index, 0], scores=scores_pre, pca=pca_pre, batch=batch, outlier_table=outlier_pre, color_map=color_map, title=f'{modality} {ion_mode}: Before ComBat', batch_r2=r2_pre)
        plot_pca_panel(ax=axes[row_index, 1], scores=scores_post, pca=pca_post, batch=batch, outlier_table=outlier_post, color_map=color_map, title=f'{modality} {ion_mode}: After ComBat', batch_r2=r2_post)
    fig.suptitle(f'{modality} metabolomics: batch-correction acceptance PCA', fontsize=15, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.975])
    png_path = OUT_DIR / f"{modality}{get_value('science.figures.plot_098', 'plot_098.png')}"
    pdf_path = OUT_DIR / f"{modality}{get_value('science.figures.plot_099', 'plot_099.pdf')}"
    fig.savefig(png_path, dpi=FIG_DPI, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return (pd.DataFrame(summary_records), pd.concat(score_tables, ignore_index=True), pd.concat(outlier_tables, ignore_index=True))
all_summary = []
all_scores = []
all_outliers = []
for modality, config in FILES.items():
    summary, scores, outliers = analyze_modality(modality, config)
    all_summary.append(summary)
    all_scores.append(scores)
    all_outliers.append(outliers)
summary_df = pd.concat(all_summary, ignore_index=True)
scores_df = pd.concat(all_scores, ignore_index=True)
outlier_diagnostics_df = pd.concat(all_outliers, ignore_index=True)
flagged_outliers_df = outlier_diagnostics_df.loc[outlier_diagnostics_df['Outlier']].copy().sort_values(['Modality', 'Ion mode', 'Stage', 'Robust distance squared'], ascending=[True, True, True, False])
summary_path = OUT_DIR / get_value('science.tables.metrics_063', 'metrics_063.csv')
scores_path = OUT_DIR / get_value('science.tables.result_159', 'result_159.csv')
outlier_diagnostics_path = OUT_DIR / get_value('science.tables.result_160', 'result_160.csv')
flagged_outliers_path = OUT_DIR / get_value('science.tables.result_161', 'result_161.csv')
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
scores_df.to_csv(scores_path, index=False, encoding='utf-8-sig')
outlier_diagnostics_df.to_csv(outlier_diagnostics_path, index=False, encoding='utf-8-sig')
flagged_outliers_df.to_csv(flagged_outliers_path, index=False, encoding='utf-8-sig')

print('ComBat minimal validation summary')

display_columns = ['Modality', 'Ion mode', 'N samples', 'N features', 'N batches', 'Batch R2 before', 'Batch R2 after', 'Batch R2 retained proportion', 'Within-batch distance Spearman rho', 'Median feature SD ratio post/pre', 'Outliers before', 'Outliers after', 'QC flag (heuristic)']
try:
    from IPython.display import display
    
    print('\nFlagged clear outlier samples:')
    if flagged_outliers_df.empty:
        print('No clear outlier samples met the strict threshold.')
    else:
        pass
except ImportError:
    print(summary_df[display_columns].round(4).to_string(index=False))
    print('\nFlagged clear outlier samples:')
    if flagged_outliers_df.empty:
        print('No clear outlier samples met the strict threshold.')
    else:
        print(flagged_outliers_df.to_string(index=False))






print(get_value('science.figures.plot_100', 'plot_100.pdf'))
print(get_value('science.figures.plot_101', 'plot_101.pdf'))
